# Week 1 — 2026 NFL Game Projections

This notebook generates Week 1 projections for the 2026 NFL season.

The goal is to build on the completed preseason projection model and produce:

- Projected final scores
- Straight up winners
- Against the spread picks
- Over/under picks
- Model edges versus the betting market

Week 1 uses the preseason team strength ratings as the primary team quality input because no 2026 regular season games have been played yet.

In [182]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm

In [183]:
PROJECT_ROOT = Path("../..")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

WEEKLY_OUTPUT_DIR = (
    PROJECT_ROOT
    / "weekly_projections"
    / "outputs"
)

WEEKLY_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

WEEKLY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [184]:
team_strength_2026 = pd.read_parquet(
    PROCESSED_DIR / "2026_team_strength.parquet"
)

game_predictions_2026 = pd.read_parquet(
    PROCESSED_DIR / "2026_game_predictions.parquet"
)

print(
    "Team Strength:",
    team_strength_2026.shape
)

print(
    "Game Predictions:",
    game_predictions_2026.shape
)

display(
    team_strength_2026.head()
)

display(
    game_predictions_2026.head()
)

Team Strength: (32, 8)
Game Predictions: (272, 16)


,strength_rank,team,team_strength,baseline_team_strength,personnel_adjustment,personnel_strength,roster_continuity,roster_continuity_adjustment
0,1,BUF,4.816514,4.125764,0.934506,0.623004,0.573034,0.260091
1,2,LA,4.762452,3.257133,2.100811,1.400540,0.608696,0.489665
2,3,SEA,4.363421,3.515976,0.635683,0.423788,0.677419,0.932073
3,4,DET,4.018565,3.374644,1.041923,0.694616,0.538462,0.037533
4,5,DEN,3.569507,2.355154,0.979753,0.653169,0.727273,1.253003


,game_id,week,gameday,away_team,home_team,away_team_strength,home_team_strength,neutral_strength_diff,home_field_adjustment,rest_diff,rest_adjustment,expected_home_margin,away_win_probability,home_win_probability,predicted_winner,predicted_win_probability
0,2026_01_NE_SEA,1,2026-09-09,NE,SEA,1.910188,4.363421,2.453233,1.763528,0,0.0,4.216761,0.383257,0.616743,SEA,0.616743
1,2026_01_SF_LA,1,2026-09-10,SF,LA,2.366788,4.762452,2.395664,0.000000,0,0.0,2.395664,0.433017,0.566983,LA,0.566983
2,2026_01_CHI_CAR,1,2026-09-13,CHI,CAR,-0.051934,-4.533242,-4.481308,1.763528,0,0.0,-2.717780,0.575887,0.424113,CHI,0.575887
3,2026_01_TB_CIN,1,2026-09-13,TB,CIN,0.601026,-0.739434,-1.340461,1.763528,0,0.0,0.423067,0.488117,0.511883,CIN,0.511883
4,2026_01_NO_DET,1,2026-09-13,NO,DET,-1.556053,4.018565,5.574618,1.763528,0,0.0,7.338146,0.302668,0.697332,DET,0.697332


# Week 1 Schedule & Preseason Game Projections

The completed preseason model already contains an expected scoring margin and win probability for every 2026 regular season game.

For Week 1, these existing predictions serve as the starting point for the weekly projection model. The next step will be adding an independent total points projection so that expected margins can be converted into projected final scores.

In [185]:
week_1 = (
    game_predictions_2026[
        game_predictions_2026["week"] == 1
    ]
    .copy()
    .sort_values(
        ["gameday", "game_id"]
    )
    .reset_index(drop=True)
)

print(
    "Week 1 Games:",
    len(week_1)
)

display(
    week_1[
        [
            "gameday",
            "away_team",
            "home_team",
            "away_team_strength",
            "home_team_strength",
            "home_field_adjustment",
            "rest_adjustment",
            "expected_home_margin",
            "predicted_winner",
            "predicted_win_probability"
        ]
    ]
)

Week 1 Games: 16


,gameday,away_team,home_team,away_team_strength,home_team_strength,home_field_adjustment,rest_adjustment,expected_home_margin,predicted_winner,predicted_win_probability
0,2026-09-09,NE,SEA,1.910188,4.363421,1.763528,0.0,4.216761,SEA,0.616743
1,2026-09-10,SF,LA,2.366788,4.762452,0.000000,0.0,2.395664,LA,0.566983
2,2026-09-13,ARI,LAC,-3.678016,0.910681,1.763528,0.0,6.352224,LAC,0.672676
3,2026-09-13,ATL,PIT,-1.331272,0.543902,1.763528,0.0,3.638702,PIT,0.601114
4,2026-09-13,BAL,IND,3.074111,0.532226,1.763528,0.0,-0.778357,BAL,0.521855
5,2026-09-13,BUF,HOU,4.816514,2.810133,1.763528,0.0,-0.242853,BUF,0.506822
6,2026-09-13,CHI,CAR,-0.051934,-4.533242,1.763528,0.0,-2.717780,CHI,0.575887
7,2026-09-13,CLE,JAX,-4.150395,1.713136,1.763528,0.0,7.627059,JAX,0.704396
8,2026-09-13,DAL,NYG,-0.386396,-3.455862,1.763528,0.0,-1.305937,DAL,0.536636
9,2026-09-13,GB,MIN,2.536317,0.475040,1.763528,0.0,-0.297749,GB,0.508364


In [186]:
week_1.to_parquet(
    WEEKLY_DATA_DIR / "week_01_preseason_predictions.parquet",
    index=False
)

# Historical Scoring Data

Projected final scores require two separate estimates:

1. Expected scoring margin
2. Expected combined points

The preseason model already provides the expected margin. The next step is to examine the historical team data available for building an independent total points model.

In [187]:
historical_team_features = pd.read_parquet(
    PROCESSED_DIR / "historical_team_features.parquet"
)

print("Shape:", historical_team_features.shape)

print("\nColumns:")
print(historical_team_features.columns.tolist())

display(
    historical_team_features.head()
)

Shape: (352, 156)

Columns:
['season', 'team', 'games', 'wins', 'losses', 'ties', 'points_for', 'points_against', 'point_diff', 'win_pct', 'points_for_per_game', 'points_against_per_game', 'point_diff_per_game', 'win_loss_diff', 'positive_point_diff', 'one_score_games', 'one_score_wins', 'one_score_losses', 'one_score_ties', 'one_score_win_pct', 'off_plays', 'off_epa_per_play', 'off_success_rate', 'off_pass_epa_per_play', 'off_rush_epa_per_play', 'def_plays', 'def_epa_per_play_allowed', 'def_success_rate_allowed', 'def_pass_epa_per_play_allowed', 'def_rush_epa_per_play_allowed', 'off_explosive_rate', 'off_explosive_pass_rate', 'off_explosive_rush_rate', 'def_explosive_rate_allowed', 'def_explosive_pass_rate_allowed', 'def_explosive_rush_rate_allowed', 'interceptions_thrown', 'fumbles_lost', 'turnovers', 'def_interceptions', 'def_fumble_recoveries', 'takeaways', 'def_pass_plays', 'sacks', 'qb_hits', 'sack_rate', 'qb_hit_rate', 'turnover_margin', 'third_down_conversions', 'third_down_att

,season,team,games,wins,losses,ties,points_for,points_against,point_diff,win_pct,...,road_games,home_wins,road_wins,home_avg_point_diff,road_avg_point_diff,home_win_pct,road_win_pct,avg_rest_days,short_rest_games,extended_rest_games
0,2015,ARI,16,13,3,0,489,313,176,0.8125,...,8,6,7,8.000,14.000,0.750,0.875,7.4375,2,3
1,2015,ATL,16,8,8,0,339,345,-6,0.5000,...,8,4,4,2.875,-3.625,0.500,0.500,7.3750,2,2
2,2015,BAL,16,5,11,0,328,401,-73,0.3125,...,8,3,2,-6.125,-3.000,0.375,0.250,7.4375,3,4
3,2015,BUF,16,8,8,0,379,359,20,0.5000,...,8,5,3,2.250,0.250,0.625,0.375,7.4375,2,2
4,2015,CAR,16,15,1,0,500,308,192,0.9375,...,8,8,7,16.000,8.000,1.000,0.875,7.4375,2,3


In [188]:
scoring_columns = [
    col for col in historical_team_features.columns
    if any(
        term in col.lower()
        for term in [
            "point",
            "score",
            "offense",
            "defense",
            "total"
        ]
    )
]

print("Potential scoring-related columns:")
print(scoring_columns)

Potential scoring-related columns:
['points_for', 'points_against', 'point_diff', 'points_for_per_game', 'points_against_per_game', 'point_diff_per_game', 'positive_point_diff', 'one_score_games', 'one_score_wins', 'one_score_losses', 'one_score_ties', 'one_score_win_pct', 'offense_continuity', 'defense_continuity', 'one_score_wins_above_500', 'prev_one_score_win_pct', 'prev_one_score_wins_above_500', 'off_drive_points', 'points_per_drive', 'def_drive_points_allowed', 'points_per_drive_allowed', 'home_avg_point_diff', 'road_avg_point_diff']


# Total Points Model

The preseason model estimates the expected margin for each game, but a projected final score also requires an estimate of total points scored.

To build this component, historical team scoring and defensive performance will be used to estimate how offensive scoring carries forward from one season to the next. The model will be evaluated using walk forward validation so that each season is predicted using only information that would have been available beforehand.

In [189]:
scoring_history = (
    historical_team_features[
        [
            "season",
            "team",
            "points_for_per_game",
            "points_against_per_game"
        ]
    ]
    .copy()
    .sort_values(["team", "season"])
)

scoring_history["prev_points_for_per_game"] = (
    scoring_history
    .groupby("team")["points_for_per_game"]
    .shift(1)
)

scoring_history["prev_points_against_per_game"] = (
    scoring_history
    .groupby("team")["points_against_per_game"]
    .shift(1)
)

scoring_history = scoring_history.dropna().reset_index(drop=True)

display(scoring_history.head(10))

,season,team,points_for_per_game,points_against_per_game,prev_points_for_per_game,prev_points_against_per_game
0,2016,ARI,26.125000,22.625000,30.562500,19.562500
1,2017,ARI,18.437500,22.562500,26.125000,22.625000
2,2018,ARI,14.062500,26.562500,18.437500,22.562500
3,2019,ARI,22.562500,27.625000,14.062500,26.562500
4,2020,ARI,25.625000,22.937500,22.562500,27.625000
5,2021,ARI,26.411765,21.529412,25.625000,22.937500
6,2022,ARI,20.000000,26.411765,26.411765,21.529412
7,2023,ARI,19.411765,26.764706,20.000000,26.411765
8,2024,ARI,23.529412,22.294118,19.411765,26.764706
9,2025,ARI,20.882353,28.705882,23.529412,22.294118


In [190]:
offense_correlation = scoring_history[
    [
        "prev_points_for_per_game",
        "points_for_per_game"
    ]
].corr().iloc[0, 1]

defense_correlation = scoring_history[
    [
        "prev_points_against_per_game",
        "points_against_per_game"
    ]
].corr().iloc[0, 1]

print(
    f"Year-to-Year Offensive Scoring Correlation: "
    f"{offense_correlation:.3f}"
)

print(
    f"Year-to-Year Defensive Scoring Correlation: "
    f"{defense_correlation:.3f}"
)

Year-to-Year Offensive Scoring Correlation: 0.363
Year-to-Year Defensive Scoring Correlation: 0.295


In [191]:
drive_history = (
    historical_team_features[
        [
            "season",
            "team",
            "points_per_drive",
            "points_per_drive_allowed"
        ]
    ]
    .copy()
    .sort_values(["team", "season"])
)

drive_history["prev_points_per_drive"] = (
    drive_history
    .groupby("team")["points_per_drive"]
    .shift(1)
)

drive_history["prev_points_per_drive_allowed"] = (
    drive_history
    .groupby("team")["points_per_drive_allowed"]
    .shift(1)
)

drive_history = drive_history.dropna().reset_index(drop=True)

off_drive_correlation = drive_history[
    [
        "prev_points_per_drive",
        "points_per_drive"
    ]
].corr().iloc[0, 1]

def_drive_correlation = drive_history[
    [
        "prev_points_per_drive_allowed",
        "points_per_drive_allowed"
    ]
].corr().iloc[0, 1]

print(
    f"Year-to-Year Offensive Points/Drive Correlation: "
    f"{off_drive_correlation:.3f}"
)

print(
    f"Year-to-Year Defensive Points/Drive Correlation: "
    f"{def_drive_correlation:.3f}"
)

Year-to-Year Offensive Points/Drive Correlation: 0.409
Year-to-Year Defensive Points/Drive Correlation: 0.309


# Historical Game Level Total Points

Season level scoring metrics provide information about offensive and defensive quality, but the final target is the combined score of an individual game.

Historical games are paired with each team's prior season offensive and defensive efficiency. This creates a game level dataset that can be used to predict total points without using information from the season being predicted.

In [192]:
import nflreadpy as nfl

historical_schedule = nfl.load_schedules(
    seasons=list(range(2016, 2026))
).to_pandas()

historical_schedule = historical_schedule[
    historical_schedule["game_type"] == "REG"
].copy()

historical_schedule["total_points"] = (
    historical_schedule["home_score"]
    + historical_schedule["away_score"]
)

historical_schedule = historical_schedule.dropna(
    subset=[
        "home_score",
        "away_score",
        "total_points"
    ]
)

print("Historical Games:", len(historical_schedule))

display(
    historical_schedule[
        [
            "season",
            "week",
            "away_team",
            "home_team",
            "away_score",
            "home_score",
            "total_points"
        ]
    ].head()
)

Historical Games: 2639


,season,week,away_team,home_team,away_score,home_score,total_points
0,2016,1,CAR,DEN,20,21,41
1,2016,1,TB,ATL,31,24,55
2,2016,1,BUF,BAL,7,13,20
3,2016,1,CHI,HOU,14,23,37
4,2016,1,GB,JAX,27,23,50


In [193]:
prior_efficiency = (
    historical_team_features[
        [
            "season",
            "team",
            "points_per_drive",
            "points_per_drive_allowed"
        ]
    ]
    .copy()
)

prior_efficiency["season"] = (
    prior_efficiency["season"] + 1
)

prior_efficiency = prior_efficiency.rename(
    columns={
        "points_per_drive": "prior_off_ppd",
        "points_per_drive_allowed": "prior_def_ppd_allowed"
    }
)

display(prior_efficiency.head())

,season,team,prior_off_ppd,prior_def_ppd_allowed
0,2016,ARI,2.340426,1.641711
1,2016,ATL,1.828571,1.924419
2,2016,BAL,1.609626,1.914439
3,2016,BUF,1.888298,1.760204
4,2016,CAR,2.383838,1.469697


In [194]:
home_efficiency = prior_efficiency.rename(
    columns={
        "team": "home_team",
        "prior_off_ppd": "home_prior_off_ppd",
        "prior_def_ppd_allowed": "home_prior_def_ppd_allowed"
    }
)

away_efficiency = prior_efficiency.rename(
    columns={
        "team": "away_team",
        "prior_off_ppd": "away_prior_off_ppd",
        "prior_def_ppd_allowed": "away_prior_def_ppd_allowed"
    }
)

total_model_data = (
    historical_schedule
    .merge(
        home_efficiency,
        on=["season", "home_team"],
        how="left"
    )
    .merge(
        away_efficiency,
        on=["season", "away_team"],
        how="left"
    )
)

total_model_data = total_model_data.dropna(
    subset=[
        "home_prior_off_ppd",
        "home_prior_def_ppd_allowed",
        "away_prior_off_ppd",
        "away_prior_def_ppd_allowed"
    ]
).reset_index(drop=True)

print("Modeling Games:", len(total_model_data))

display(
    total_model_data[
        [
            "season",
            "away_team",
            "home_team",
            "total_points",
            "home_prior_off_ppd",
            "home_prior_def_ppd_allowed",
            "away_prior_off_ppd",
            "away_prior_def_ppd_allowed"
        ]
    ].head(10)
)

Modeling Games: 2561


,season,away_team,home_team,total_points,home_prior_off_ppd,home_prior_def_ppd_allowed,away_prior_off_ppd,away_prior_def_ppd_allowed
0,2016,CAR,DEN,41,1.547264,1.356436,2.383838,1.469697
1,2016,TB,ATL,55,1.828571,1.924419,1.807910,2.181319
2,2016,BUF,BAL,20,1.609626,1.914439,1.888298,1.760204
3,2016,CHI,HOU,37,1.519608,1.487562,1.830601,2.033520
4,2016,GB,JAX,50,1.798969,2.005128,1.802083,1.621622
5,2016,CIN,NYJ,45,1.892157,1.435897,2.227027,1.433333
6,2016,CLE,PHI,39,1.590244,1.951456,1.433333,2.213115
7,2016,MIN,TEN,41,1.484375,2.037433,1.920455,1.685393
8,2016,MIA,SEA,22,2.250000,1.448864,1.502674,1.895833
9,2016,NYG,DAL,39,1.497110,1.888268,2.005263,2.186528


# Total Points Model Validation

The first total points model uses prior-season offensive and defensive points per drive for both teams.

Performance is evaluated using walk forward validation. Each season is predicted using only games from earlier seasons, preventing future information from leaking into the model.

The model is compared with a simple baseline that predicts every game at the historical average total points available before that season.

In [195]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

TOTAL_FEATURES = [
    "home_prior_off_ppd",
    "home_prior_def_ppd_allowed",
    "away_prior_off_ppd",
    "away_prior_def_ppd_allowed"
]

TOTAL_TARGET = "total_points"

In [196]:
total_validation_results = []

for test_season in range(2019, 2026):

    train = total_model_data[
        total_model_data["season"] < test_season
    ].copy()

    test = total_model_data[
        total_model_data["season"] == test_season
    ].copy()

    X_train = train[TOTAL_FEATURES]
    y_train = train[TOTAL_TARGET]

    X_test = test[TOTAL_FEATURES]
    y_test = test[TOTAL_TARGET]

    total_model = LinearRegression()
    total_model.fit(X_train, y_train)

    test["predicted_total"] = total_model.predict(X_test)

    baseline_total = y_train.mean()

    test["baseline_total"] = baseline_total

    model_mae = mean_absolute_error(
        y_test,
        test["predicted_total"]
    )

    baseline_mae = mean_absolute_error(
        y_test,
        test["baseline_total"]
    )

    model_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test["predicted_total"]
        )
    )

    correlation = test[
        ["predicted_total", "total_points"]
    ].corr().iloc[0, 1]

    total_validation_results.append(
        {
            "season": test_season,
            "games": len(test),
            "model_mae": model_mae,
            "baseline_mae": baseline_mae,
            "rmse": model_rmse,
            "correlation": correlation
        }
    )

total_validation = pd.DataFrame(
    total_validation_results
)

display(
    total_validation.round(3)
)

,season,games,model_mae,baseline_mae,rmse,correlation
0,2019,240,11.117,11.267,13.851,0.193
1,2020,256,11.111,11.323,14.077,0.119
2,2021,272,11.742,11.101,14.335,0.047
3,2022,271,11.395,11.315,14.020,0.142
4,2023,272,10.900,11.024,13.719,0.116
5,2024,272,9.925,10.044,12.955,0.172
6,2025,272,10.792,10.970,13.580,0.208


In [197]:
avg_model_mae = total_validation["model_mae"].mean()
avg_baseline_mae = total_validation["baseline_mae"].mean()
avg_correlation = total_validation["correlation"].mean()

improvement = (
    (
        avg_baseline_mae
        - avg_model_mae
    )
    / avg_baseline_mae
) * 100

print(
    f"Average Model MAE: {avg_model_mae:.2f} points"
)

print(
    f"Average Baseline MAE: {avg_baseline_mae:.2f} points"
)

print(
    f"Average Improvement: {improvement:.1f}%"
)

print(
    f"Average Correlation: {avg_correlation:.3f}"
)

Average Model MAE: 11.00 points
Average Baseline MAE: 11.01 points
Average Improvement: 0.1%
Average Correlation: 0.142


# Offensive and Defensive Feature Inspection

The initial total points model provided almost no improvement over a league average baseline.

Rather than forcing prior season scoring efficiency into the model, the next step is to identify the offensive and defensive information already available in the historical feature set. These variables may provide a better representation of how individual matchups influence scoring.

In [198]:
feature_keywords = [
    "pass",
    "rush",
    "epa",
    "success",
    "drive",
    "yard",
    "turnover",
    "sack",
    "pressure",
    "play",
    "off",
    "def"
]

potential_features = [
    col for col in historical_team_features.columns
    if any(
        keyword in col.lower()
        for keyword in feature_keywords
    )
]

print(f"Potential Features: {len(potential_features)}")

for col in potential_features:
    print(col)

Potential Features: 75
off_plays
off_epa_per_play
off_success_rate
off_pass_epa_per_play
off_rush_epa_per_play
def_plays
def_epa_per_play_allowed
def_success_rate_allowed
def_pass_epa_per_play_allowed
def_rush_epa_per_play_allowed
off_explosive_rate
off_explosive_pass_rate
off_explosive_rush_rate
def_explosive_rate_allowed
def_explosive_pass_rate_allowed
def_explosive_rush_rate_allowed
turnovers
def_interceptions
def_fumble_recoveries
def_pass_plays
sacks
sack_rate
turnover_margin
off_third_down_rate
off_fourth_down_rate
def_third_down_rate_allowed
def_fourth_down_rate_allowed
off_red_zone_plays
off_red_zone_epa_per_play
off_red_zone_success_rate
def_red_zone_plays
def_red_zone_epa_per_play_allowed
def_red_zone_success_rate_allowed
offense_continuity
defense_continuity
passing_yards_continuity
rushing_yards_continuity
receiving_yards_continuity
players_on_injury_report
prev_off_epa_per_play
prev_def_epa_per_play_allowed
off_epa_change
def_epa_allowed_change
off_epa_2yr_avg
def_epa_allo

In [199]:
print(
    historical_team_features.columns.tolist()
)

['season', 'team', 'games', 'wins', 'losses', 'ties', 'points_for', 'points_against', 'point_diff', 'win_pct', 'points_for_per_game', 'points_against_per_game', 'point_diff_per_game', 'win_loss_diff', 'positive_point_diff', 'one_score_games', 'one_score_wins', 'one_score_losses', 'one_score_ties', 'one_score_win_pct', 'off_plays', 'off_epa_per_play', 'off_success_rate', 'off_pass_epa_per_play', 'off_rush_epa_per_play', 'def_plays', 'def_epa_per_play_allowed', 'def_success_rate_allowed', 'def_pass_epa_per_play_allowed', 'def_rush_epa_per_play_allowed', 'off_explosive_rate', 'off_explosive_pass_rate', 'off_explosive_rush_rate', 'def_explosive_rate_allowed', 'def_explosive_pass_rate_allowed', 'def_explosive_rush_rate_allowed', 'interceptions_thrown', 'fumbles_lost', 'turnovers', 'def_interceptions', 'def_fumble_recoveries', 'takeaways', 'def_pass_plays', 'sacks', 'qb_hits', 'sack_rate', 'qb_hit_rate', 'turnover_margin', 'third_down_conversions', 'third_down_attempts', 'fourth_down_convers

# EPA-Based Total Points Model

The initial points per drive model provided almost no improvement over a league-average scoring baseline.

The next model tests whether broader offensive and defensive efficiency measures provide stronger predictive information. Prior season EPA metrics are used for both teams to capture offensive scoring ability and defensive resistance while maintaining a relatively simple and interpretable model.

In [200]:
epa_history = (
    historical_team_features[
        [
            "season",
            "team",
            "adjusted_off_epa",
            "adjusted_def_epa_allowed"
        ]
    ]
    .copy()
)

epa_history["season"] = (
    epa_history["season"] + 1
)

epa_history = epa_history.rename(
    columns={
        "adjusted_off_epa": "prior_adjusted_off_epa",
        "adjusted_def_epa_allowed": "prior_adjusted_def_epa_allowed"
    }
)

display(epa_history.head())

,season,team,prior_adjusted_off_epa,prior_adjusted_def_epa_allowed
0,2016,ARI,0.115264,-0.082612
1,2016,ATL,-0.017568,0.010799
2,2016,BAL,-0.049901,0.031825
3,2016,BUF,0.059219,0.037767
4,2016,CAR,0.056132,-0.120934


In [201]:
home_epa = epa_history.rename(
    columns={
        "team": "home_team",
        "prior_adjusted_off_epa": "home_prior_off_epa",
        "prior_adjusted_def_epa_allowed": "home_prior_def_epa"
    }
)

away_epa = epa_history.rename(
    columns={
        "team": "away_team",
        "prior_adjusted_off_epa": "away_prior_off_epa",
        "prior_adjusted_def_epa_allowed": "away_prior_def_epa"
    }
)

epa_total_data = (
    historical_schedule
    .merge(
        home_epa,
        on=["season", "home_team"],
        how="left"
    )
    .merge(
        away_epa,
        on=["season", "away_team"],
        how="left"
    )
)

epa_total_data = epa_total_data.dropna(
    subset=[
        "home_prior_off_epa",
        "home_prior_def_epa",
        "away_prior_off_epa",
        "away_prior_def_epa"
    ]
).reset_index(drop=True)

print("EPA Modeling Games:", len(epa_total_data))

display(
    epa_total_data[
        [
            "season",
            "away_team",
            "home_team",
            "total_points",
            "home_prior_off_epa",
            "home_prior_def_epa",
            "away_prior_off_epa",
            "away_prior_def_epa"
        ]
    ].head(10)
)

EPA Modeling Games: 2561


,season,away_team,home_team,total_points,home_prior_off_epa,home_prior_def_epa,away_prior_off_epa,away_prior_def_epa
0,2016,CAR,DEN,41,-0.061198,-0.149860,0.056132,-0.120934
1,2016,TB,ATL,55,-0.017568,0.010799,0.002964,0.052049
2,2016,BUF,BAL,20,-0.049901,0.031825,0.059219,0.037767
3,2016,CHI,HOU,37,-0.054960,-0.102913,0.011544,0.056030
4,2016,GB,JAX,50,-0.055727,0.062042,-0.003488,-0.033562
5,2016,CIN,NYJ,45,0.005233,-0.082473,0.101676,-0.030983
6,2016,CLE,PHI,39,-0.059644,0.013132,-0.047822,0.080648
7,2016,MIN,TEN,41,-0.113500,0.053491,0.020404,-0.027963
8,2016,MIA,SEA,22,0.129731,-0.063867,-0.027944,0.066638
9,2016,NYG,DAL,39,-0.087903,0.035505,-0.000268,0.063904


In [202]:
EPA_TOTAL_FEATURES = [
    "home_prior_off_epa",
    "home_prior_def_epa",
    "away_prior_off_epa",
    "away_prior_def_epa"
]

epa_validation_results = []

for test_season in range(2019, 2026):

    train = epa_total_data[
        epa_total_data["season"] < test_season
    ].copy()

    test = epa_total_data[
        epa_total_data["season"] == test_season
    ].copy()

    X_train = train[EPA_TOTAL_FEATURES]
    y_train = train["total_points"]

    X_test = test[EPA_TOTAL_FEATURES]
    y_test = test["total_points"]

    model = LinearRegression()
    model.fit(X_train, y_train)

    test["predicted_total"] = model.predict(X_test)

    baseline_total = y_train.mean()
    test["baseline_total"] = baseline_total

    model_mae = mean_absolute_error(
        y_test,
        test["predicted_total"]
    )

    baseline_mae = mean_absolute_error(
        y_test,
        test["baseline_total"]
    )

    correlation = test[
        ["predicted_total", "total_points"]
    ].corr().iloc[0, 1]

    epa_validation_results.append(
        {
            "season": test_season,
            "games": len(test),
            "model_mae": model_mae,
            "baseline_mae": baseline_mae,
            "correlation": correlation
        }
    )

epa_validation = pd.DataFrame(
    epa_validation_results
)

display(epa_validation.round(3))

,season,games,model_mae,baseline_mae,correlation
0,2019,240,11.189,11.267,0.120
1,2020,256,11.202,11.323,0.110
2,2021,272,11.144,11.101,0.054
3,2022,271,11.314,11.315,0.063
4,2023,272,11.087,11.024,0.008
5,2024,272,9.990,10.044,0.168
6,2025,272,10.798,10.970,0.182


In [203]:
comparison = (
    total_validation[
        [
            "season",
            "model_mae",
            "baseline_mae",
            "correlation"
        ]
    ]
    .rename(
        columns={
            "model_mae": "ppd_mae",
            "correlation": "ppd_correlation"
        }
    )
    .merge(
        epa_validation[
            [
                "season",
                "model_mae",
                "correlation"
            ]
        ].rename(
            columns={
                "model_mae": "epa_mae",
                "correlation": "epa_correlation"
            }
        ),
        on="season"
    )
)

display(comparison.round(3))

print(
    f"PPD Average MAE: "
    f"{comparison['ppd_mae'].mean():.2f}"
)

print(
    f"EPA Average MAE: "
    f"{comparison['epa_mae'].mean():.2f}"
)

print(
    f"Baseline Average MAE: "
    f"{comparison['baseline_mae'].mean():.2f}"
)

print()

print(
    f"PPD Average Correlation: "
    f"{comparison['ppd_correlation'].mean():.3f}"
)

print(
    f"EPA Average Correlation: "
    f"{comparison['epa_correlation'].mean():.3f}"
)

,season,ppd_mae,baseline_mae,ppd_correlation,epa_mae,epa_correlation
0,2019,11.117,11.267,0.193,11.189,0.120
1,2020,11.111,11.323,0.119,11.202,0.110
2,2021,11.742,11.101,0.047,11.144,0.054
3,2022,11.395,11.315,0.142,11.314,0.063
4,2023,10.900,11.024,0.116,11.087,0.008
5,2024,9.925,10.044,0.172,9.990,0.168
6,2025,10.792,10.970,0.208,10.798,0.182


PPD Average MAE: 11.00
EPA Average MAE: 10.96
Baseline Average MAE: 11.01

PPD Average Correlation: 0.142
EPA Average Correlation: 0.101


# Matchup-Based Total Points Model

The prior season points per drive and EPA models produced little improvement over a league average scoring baseline.

Rather than treating the four team efficiency measures as independent inputs, the next approach explicitly models each offense against its opposing defense. Offensive and defensive efficiency are combined to estimate each team's expected scoring efficiency, while drive volume is incorporated to represent the expected number of scoring opportunities in the game.

In [204]:
matchup_history = (
    historical_team_features[
        [
            "season",
            "team",
            "games",
            "off_drives",
            "points_per_drive",
            "points_per_drive_allowed"
        ]
    ]
    .copy()
)

matchup_history["drives_per_game"] = (
    matchup_history["off_drives"]
    / matchup_history["games"]
)

display(
    matchup_history[
        [
            "season",
            "team",
            "points_per_drive",
            "points_per_drive_allowed",
            "drives_per_game"
        ]
    ].head()
)

,season,team,points_per_drive,points_per_drive_allowed,drives_per_game
0,2015,ARI,2.340426,1.641711,11.7500
1,2015,ATL,1.828571,1.924419,10.9375
2,2015,BAL,1.609626,1.914439,11.6875
3,2015,BUF,1.888298,1.760204,11.7500
4,2015,CAR,2.383838,1.469697,12.3750


In [205]:
league_environment = (
    matchup_history
    .groupby("season")
    .agg(
        league_ppd=(
            "points_per_drive",
            "mean"
        ),
        league_drives_per_game=(
            "drives_per_game",
            "mean"
        )
    )
    .reset_index()
)

display(
    league_environment.round(3)
)

,season,league_ppd,league_drives_per_game
0,2015,1.813,11.775
1,2016,1.887,11.541
2,2017,1.759,11.629
3,2018,1.974,11.312
4,2019,1.909,11.309
5,2020,2.174,10.988
6,2021,2.030,10.895
7,2022,1.915,10.972
8,2023,1.881,11.114
9,2024,2.059,10.783


In [206]:
matchup_history = matchup_history.merge(
    league_environment,
    on="season",
    how="left"
)

matchup_history["off_ppd_above_avg"] = (
    matchup_history["points_per_drive"]
    - matchup_history["league_ppd"]
)

matchup_history["def_ppd_above_avg"] = (
    matchup_history["points_per_drive_allowed"]
    - matchup_history["league_ppd"]
)

display(
    matchup_history[
        [
            "season",
            "team",
            "off_ppd_above_avg",
            "def_ppd_above_avg",
            "drives_per_game"
        ]
    ].head(10)
)

,season,team,off_ppd_above_avg,def_ppd_above_avg,drives_per_game
0,2015,ARI,0.527773,-0.170941,11.7500
1,2015,ATL,0.015919,0.111766,10.9375
2,2015,BAL,-0.203027,0.101786,11.6875
3,2015,BUF,0.075645,-0.052448,11.7500
4,2015,CAR,0.571186,-0.342956,12.3750
5,2015,CHI,0.017949,0.220867,11.4375
6,2015,CIN,0.414375,-0.379319,11.5625
7,2015,CLE,-0.379319,0.400462,11.2500
8,2015,DAL,-0.315543,0.075616,10.8125
9,2015,DEN,-0.265389,-0.456217,12.5625


In [207]:
prior_matchup = (
    matchup_history[
        [
            "season",
            "team",
            "off_ppd_above_avg",
            "def_ppd_above_avg",
            "drives_per_game"
        ]
    ]
    .copy()
)

prior_matchup["season"] += 1

prior_matchup = prior_matchup.rename(
    columns={
        "off_ppd_above_avg": "prior_off_ppd_above_avg",
        "def_ppd_above_avg": "prior_def_ppd_above_avg",
        "drives_per_game": "prior_drives_per_game"
    }
)

In [208]:
home_matchup = prior_matchup.rename(
    columns={
        "team": "home_team",
        "prior_off_ppd_above_avg": "home_prior_off_ppd_above_avg",
        "prior_def_ppd_above_avg": "home_prior_def_ppd_above_avg",
        "prior_drives_per_game": "home_prior_drives_per_game"
    }
)

away_matchup = prior_matchup.rename(
    columns={
        "team": "away_team",
        "prior_off_ppd_above_avg": "away_prior_off_ppd_above_avg",
        "prior_def_ppd_above_avg": "away_prior_def_ppd_above_avg",
        "prior_drives_per_game": "away_prior_drives_per_game"
    }
)

matchup_total_data = (
    historical_schedule
    .merge(
        home_matchup,
        on=["season", "home_team"],
        how="left"
    )
    .merge(
        away_matchup,
        on=["season", "away_team"],
        how="left"
    )
)

matchup_total_data = matchup_total_data.dropna(
    subset=[
        "home_prior_off_ppd_above_avg",
        "home_prior_def_ppd_above_avg",
        "home_prior_drives_per_game",
        "away_prior_off_ppd_above_avg",
        "away_prior_def_ppd_above_avg",
        "away_prior_drives_per_game"
    ]
).reset_index(drop=True)

print("Matchup Modeling Games:", len(matchup_total_data))

Matchup Modeling Games: 2561


In [209]:
prior_league_environment = (
    league_environment
    .copy()
)

prior_league_environment["season"] += 1

prior_league_environment = prior_league_environment.rename(
    columns={
        "league_ppd": "prior_league_ppd",
        "league_drives_per_game": "prior_league_drives_per_game"
    }
)

matchup_total_data = matchup_total_data.merge(
    prior_league_environment,
    on="season",
    how="left"
)

matchup_total_data = matchup_total_data.dropna(
    subset=[
        "prior_league_ppd",
        "prior_league_drives_per_game"
    ]
).reset_index(drop=True)

In [210]:
matchup_total_data["home_expected_ppd"] = (
    matchup_total_data["prior_league_ppd"]
    + matchup_total_data["home_prior_off_ppd_above_avg"]
    + matchup_total_data["away_prior_def_ppd_above_avg"]
)

matchup_total_data["away_expected_ppd"] = (
    matchup_total_data["prior_league_ppd"]
    + matchup_total_data["away_prior_off_ppd_above_avg"]
    + matchup_total_data["home_prior_def_ppd_above_avg"]
)

matchup_total_data["expected_drives"] = (
    matchup_total_data[
        [
            "home_prior_drives_per_game",
            "away_prior_drives_per_game"
        ]
    ]
    .mean(axis=1)
)

matchup_total_data["matchup_projected_total"] = (
    (
        matchup_total_data["home_expected_ppd"]
        + matchup_total_data["away_expected_ppd"]
    )
    * matchup_total_data["expected_drives"]
)

display(
    matchup_total_data[
        [
            "season",
            "away_team",
            "home_team",
            "total_points",
            "home_expected_ppd",
            "away_expected_ppd",
            "expected_drives",
            "matchup_projected_total"
        ]
    ].head(10).round(2)
)

,season,away_team,home_team,total_points,home_expected_ppd,away_expected_ppd,expected_drives,matchup_projected_total
0,2016,CAR,DEN,41,1.20,1.93,12.47,39.05
1,2016,TB,ATL,55,2.20,1.92,11.00,45.29
2,2016,BUF,BAL,20,1.56,1.99,11.72,41.57
3,2016,CHI,HOU,37,1.74,1.51,12.09,39.26
4,2016,GB,JAX,50,1.61,1.99,12.06,43.46
5,2016,CIN,NYJ,45,1.51,1.85,12.16,40.88
6,2016,CLE,PHI,39,1.99,1.57,12.03,42.87
7,2016,MIN,TEN,41,1.36,2.15,11.50,40.28
8,2016,MIA,SEA,22,2.33,1.14,11.34,39.39
9,2016,NYG,DAL,39,1.87,2.08,11.34,44.83


In [211]:
matchup_validation_results = []

for test_season in range(2019, 2026):

    test = matchup_total_data[
        matchup_total_data["season"] == test_season
    ].copy()

    historical_games = matchup_total_data[
        matchup_total_data["season"] < test_season
    ]

    baseline_total = (
        historical_games["total_points"].mean()
    )

    test["baseline_total"] = baseline_total

    model_mae = mean_absolute_error(
        test["total_points"],
        test["matchup_projected_total"]
    )

    baseline_mae = mean_absolute_error(
        test["total_points"],
        test["baseline_total"]
    )

    correlation = test[
        [
            "matchup_projected_total",
            "total_points"
        ]
    ].corr().iloc[0, 1]

    matchup_validation_results.append(
        {
            "season": test_season,
            "games": len(test),
            "model_mae": model_mae,
            "baseline_mae": baseline_mae,
            "correlation": correlation
        }
    )

matchup_validation = pd.DataFrame(
    matchup_validation_results
)

display(
    matchup_validation.round(3)
)

,season,games,model_mae,baseline_mae,correlation
0,2019,240,11.881,11.267,0.196
1,2020,256,12.429,11.323,0.155
2,2021,272,12.194,11.101,0.084
3,2022,271,11.330,11.315,0.133
4,2023,272,11.054,11.024,0.116
5,2024,272,11.123,10.044,0.150
6,2025,272,11.225,10.970,0.192


In [212]:
print("TOTAL MODEL COMPARISON")
print("-" * 40)

print(
    f"Baseline MAE: "
    f"{matchup_validation['baseline_mae'].mean():.2f}"
)

print(
    f"PPD Regression MAE: "
    f"{total_validation['model_mae'].mean():.2f}"
)

print(
    f"EPA Regression MAE: "
    f"{epa_validation['model_mae'].mean():.2f}"
)

print(
    f"Matchup Model MAE: "
    f"{matchup_validation['model_mae'].mean():.2f}"
)

print()

print(
    f"Matchup Model Correlation: "
    f"{matchup_validation['correlation'].mean():.3f}"
)

TOTAL MODEL COMPARISON
----------------------------------------
Baseline MAE: 11.01
PPD Regression MAE: 11.00
EPA Regression MAE: 10.96
Matchup Model MAE: 11.61

Matchup Model Correlation: 0.146


# Multi-Year Scoring Baseline

Single season offensive and defensive efficiency produced limited predictive value for game totals.

The next approach follows the same philosophy used in the preseason team strength model: combine multiple prior seasons, place the greatest weight on the most recent season, and allow older seasons to stabilize the estimate.

Separate offensive and defensive scoring ratings are created so that each Week 1 matchup can ultimately be evaluated using both teams' projected scoring ability.

In [213]:
scoring_baseline = (
    historical_team_features[
        [
            "season",
            "team",
            "points_for_per_game",
            "points_against_per_game"
        ]
    ]
    .copy()
    .sort_values(["team", "season"])
)

for lag in [1, 2, 3]:

    scoring_baseline[
        f"off_scoring_lag{lag}"
    ] = (
        scoring_baseline
        .groupby("team")["points_for_per_game"]
        .shift(lag)
    )

    scoring_baseline[
        f"def_scoring_lag{lag}"
    ] = (
        scoring_baseline
        .groupby("team")["points_against_per_game"]
        .shift(lag)
    )

display(
    scoring_baseline[
        [
            "season",
            "team",
            "points_for_per_game",
            "off_scoring_lag1",
            "off_scoring_lag2",
            "off_scoring_lag3",
            "points_against_per_game",
            "def_scoring_lag1",
            "def_scoring_lag2",
            "def_scoring_lag3"
        ]
    ].dropna().head()
)

,season,team,points_for_per_game,off_scoring_lag1,off_scoring_lag2,off_scoring_lag3,points_against_per_game,def_scoring_lag1,def_scoring_lag2,def_scoring_lag3
96,2018,ARI,14.062500,18.437500,26.1250,30.5625,26.562500,22.562500,22.6250,19.5625
128,2019,ARI,22.562500,14.062500,18.4375,26.1250,27.625000,26.562500,22.5625,22.6250
160,2020,ARI,25.625000,22.562500,14.0625,18.4375,22.937500,27.625000,26.5625,22.5625
192,2021,ARI,26.411765,25.625000,22.5625,14.0625,21.529412,22.937500,27.6250,26.5625
224,2022,ARI,20.000000,26.411765,25.6250,22.5625,26.411765,21.529412,22.9375,27.6250


In [214]:
scoring_baseline["weighted_off_scoring"] = (
    0.60 * scoring_baseline["off_scoring_lag1"]
    + 0.25 * scoring_baseline["off_scoring_lag2"]
    + 0.15 * scoring_baseline["off_scoring_lag3"]
)

scoring_baseline["weighted_def_scoring"] = (
    0.60 * scoring_baseline["def_scoring_lag1"]
    + 0.25 * scoring_baseline["def_scoring_lag2"]
    + 0.15 * scoring_baseline["def_scoring_lag3"]
)

scoring_baseline_model_data = (
    scoring_baseline
    .dropna(
        subset=[
            "weighted_off_scoring",
            "weighted_def_scoring"
        ]
    )
    .reset_index(drop=True)
)

In [215]:
scoring_validation_results = []

for test_season in range(2019, 2026):

    train = scoring_baseline_model_data[
        scoring_baseline_model_data["season"] < test_season
    ].copy()

    test = scoring_baseline_model_data[
        scoring_baseline_model_data["season"] == test_season
    ].copy()

    # Offensive scoring model
    off_model = LinearRegression()

    off_model.fit(
        train[["weighted_off_scoring"]],
        train["points_for_per_game"]
    )

    test["projected_off_scoring"] = off_model.predict(
        test[["weighted_off_scoring"]]
    )

    # Defensive scoring model
    def_model = LinearRegression()

    def_model.fit(
        train[["weighted_def_scoring"]],
        train["points_against_per_game"]
    )

    test["projected_def_scoring"] = def_model.predict(
        test[["weighted_def_scoring"]]
    )

    off_mae = mean_absolute_error(
        test["points_for_per_game"],
        test["projected_off_scoring"]
    )

    def_mae = mean_absolute_error(
        test["points_against_per_game"],
        test["projected_def_scoring"]
    )

    off_corr = test[
        [
            "projected_off_scoring",
            "points_for_per_game"
        ]
    ].corr().iloc[0, 1]

    def_corr = test[
        [
            "projected_def_scoring",
            "points_against_per_game"
        ]
    ].corr().iloc[0, 1]

    scoring_validation_results.append(
        {
            "season": test_season,
            "off_mae": off_mae,
            "def_mae": def_mae,
            "off_correlation": off_corr,
            "def_correlation": def_corr
        }
    )

scoring_validation = pd.DataFrame(
    scoring_validation_results
)

display(scoring_validation.round(3))

,season,off_mae,def_mae,off_correlation,def_correlation
0,2019,2.938,3.004,0.494,0.528
1,2020,3.318,3.591,0.433,0.063
2,2021,3.526,2.371,0.521,0.529
3,2022,3.314,2.401,0.310,0.316
4,2023,3.149,2.397,0.509,0.349
5,2024,3.358,2.663,0.433,-0.065
6,2025,3.163,2.748,0.366,0.443


In [216]:
print("MULTI-YEAR SCORING VALIDATION")
print("-" * 40)

print(
    f"Average Offensive MAE: "
    f"{scoring_validation['off_mae'].mean():.2f} points/game"
)

print(
    f"Average Defensive MAE: "
    f"{scoring_validation['def_mae'].mean():.2f} points/game"
)

print(
    f"Average Offensive Correlation: "
    f"{scoring_validation['off_correlation'].mean():.3f}"
)

print(
    f"Average Defensive Correlation: "
    f"{scoring_validation['def_correlation'].mean():.3f}"
)

MULTI-YEAR SCORING VALIDATION
----------------------------------------
Average Offensive MAE: 3.25 points/game
Average Defensive MAE: 2.74 points/game
Average Offensive Correlation: 0.438
Average Defensive Correlation: 0.309


# Projected Scoring Matchup Model

The multi year scoring baseline showed substantially stronger year to year predictive value than the single season efficiency approaches.

The projected offensive and defensive scoring ratings are now combined at the game level. Each team's expected scoring output is based on its projected offensive scoring ability and the opposing team's projected defensive scoring ability.

In [217]:
historical_scoring_projections = []

for test_season in range(2019, 2026):

    train = scoring_baseline_model_data[
        scoring_baseline_model_data["season"] < test_season
    ].copy()

    test = scoring_baseline_model_data[
        scoring_baseline_model_data["season"] == test_season
    ].copy()

    off_model = LinearRegression()
    off_model.fit(
        train[["weighted_off_scoring"]],
        train["points_for_per_game"]
    )

    def_model = LinearRegression()
    def_model.fit(
        train[["weighted_def_scoring"]],
        train["points_against_per_game"]
    )

    test["projected_off_ppg"] = off_model.predict(
        test[["weighted_off_scoring"]]
    )

    test["projected_def_ppg_allowed"] = def_model.predict(
        test[["weighted_def_scoring"]]
    )

    historical_scoring_projections.append(
        test[
            [
                "season",
                "team",
                "projected_off_ppg",
                "projected_def_ppg_allowed"
            ]
        ]
    )

historical_scoring_projections = pd.concat(
    historical_scoring_projections,
    ignore_index=True
)

display(
    historical_scoring_projections.head(10).round(2)
)

,season,team,projected_off_ppg,projected_def_ppg_allowed
0,2019,ARI,20.49,24.81
1,2019,ATL,25.52,24.61
2,2019,BAL,24.35,21.45
3,2019,BUF,21.36,23.88
4,2019,CAR,23.94,23.89
5,2019,CHI,23.55,21.89
6,2019,CIN,22.92,25.07
7,2019,CLE,21.94,25.00
8,2019,DAL,23.37,22.34
9,2019,DEN,22.15,23.18


In [218]:
home_scoring = historical_scoring_projections.rename(
    columns={
        "team": "home_team",
        "projected_off_ppg": "home_projected_off_ppg",
        "projected_def_ppg_allowed": "home_projected_def_ppg"
    }
)

away_scoring = historical_scoring_projections.rename(
    columns={
        "team": "away_team",
        "projected_off_ppg": "away_projected_off_ppg",
        "projected_def_ppg_allowed": "away_projected_def_ppg"
    }
)

scoring_game_data = (
    historical_schedule
    .merge(
        home_scoring,
        on=["season", "home_team"],
        how="inner"
    )
    .merge(
        away_scoring,
        on=["season", "away_team"],
        how="inner"
    )
)

print(
    "Historical Scoring Games:",
    len(scoring_game_data)
)

Historical Scoring Games: 1855


In [219]:
scoring_game_data["home_matchup_points"] = (
    scoring_game_data["home_projected_off_ppg"]
    + scoring_game_data["away_projected_def_ppg"]
) / 2

scoring_game_data["away_matchup_points"] = (
    scoring_game_data["away_projected_off_ppg"]
    + scoring_game_data["home_projected_def_ppg"]
) / 2

scoring_game_data["projected_total"] = (
    scoring_game_data["home_matchup_points"]
    + scoring_game_data["away_matchup_points"]
)

display(
    scoring_game_data[
        [
            "season",
            "away_team",
            "home_team",
            "total_points",
            "away_matchup_points",
            "home_matchup_points",
            "projected_total"
        ]
    ].head(10).round(2)
)

,season,away_team,home_team,total_points,away_matchup_points,home_matchup_points,projected_total
0,2019,GB,CHI,13,22.89,24.10,46.99
1,2019,LA,CAR,57,25.59,23.92,49.52
2,2019,TEN,CLE,56,23.70,22.21,45.90
3,2019,KC,JAX,66,25.25,22.87,48.13
4,2019,BAL,MIA,69,24.83,21.74,46.57
5,2019,ATL,MIN,40,23.77,24.09,47.86
6,2019,BUF,NYJ,33,23.44,22.94,46.39
7,2019,WAS,PHI,59,22.28,24.20,46.48
8,2019,IND,LAC,54,23.49,24.44,47.94
9,2019,CIN,SEA,41,22.82,25.02,47.83


In [220]:
scoring_game_validation = []

for season in range(2019, 2026):

    test = scoring_game_data[
        scoring_game_data["season"] == season
    ].copy()

    prior_games = historical_schedule[
        historical_schedule["season"] < season
    ]

    baseline_total = (
        prior_games["total_points"].mean()
    )

    test["baseline_total"] = baseline_total

    model_mae = mean_absolute_error(
        test["total_points"],
        test["projected_total"]
    )

    baseline_mae = mean_absolute_error(
        test["total_points"],
        test["baseline_total"]
    )

    correlation = test[
        ["projected_total", "total_points"]
    ].corr().iloc[0, 1]

    scoring_game_validation.append(
        {
            "season": season,
            "games": len(test),
            "model_mae": model_mae,
            "baseline_mae": baseline_mae,
            "correlation": correlation
        }
    )

scoring_game_validation = pd.DataFrame(
    scoring_game_validation
)

display(
    scoring_game_validation.round(3)
)

,season,games,model_mae,baseline_mae,correlation
0,2019,240,11.303,11.272,0.169
1,2020,256,11.223,11.301,0.003
2,2021,272,11.319,11.106,0.140
3,2022,271,11.539,11.330,0.154
4,2023,272,11.035,11.038,0.119
5,2024,272,9.963,10.044,0.122
6,2025,272,10.870,10.971,0.189


In [221]:
print("SCORING MATCHUP MODEL")
print("-" * 40)

print(
    f"Model MAE: "
    f"{scoring_game_validation['model_mae'].mean():.2f}"
)

print(
    f"Baseline MAE: "
    f"{scoring_game_validation['baseline_mae'].mean():.2f}"
)

improvement = (
    (
        scoring_game_validation["baseline_mae"].mean()
        - scoring_game_validation["model_mae"].mean()
    )
    / scoring_game_validation["baseline_mae"].mean()
) * 100

print(
    f"Improvement vs Baseline: "
    f"{improvement:.1f}%"
)

print(
    f"Average Correlation: "
    f"{scoring_game_validation['correlation'].mean():.3f}"
)

seasons_beaten = (
    scoring_game_validation["model_mae"]
    < scoring_game_validation["baseline_mae"]
).sum()

print(
    f"Baseline Beaten: "
    f"{seasons_beaten}/{len(scoring_game_validation)} seasons"
)

SCORING MATCHUP MODEL
----------------------------------------
Model MAE: 11.04
Baseline MAE: 11.01
Improvement vs Baseline: -0.2%
Average Correlation: 0.128
Baseline Beaten: 4/7 seasons


# Calibrating Team Influence on Game Totals

The previous matchup model allowed projected offensive and defensive strength to fully determine each game's scoring expectation. Historical validation showed that this produced too much noise and did not outperform a league-average baseline.

The next step tests how strongly team level scoring information should influence a game total relative to the overall NFL scoring environment.

In [222]:
season_scoring_environment = (
    historical_schedule
    .groupby("season")["total_points"]
    .mean()
    .reset_index(name="league_game_total")
)

season_scoring_environment["season"] += 1

season_scoring_environment = (
    season_scoring_environment.rename(
        columns={
            "league_game_total": "prior_league_game_total"
        }
    )
)

# Remove any previous versions created by rerunning this cell
columns_to_remove = [
    col for col in scoring_game_data.columns
    if col.startswith("prior_league_game_total")
]

scoring_game_data = scoring_game_data.drop(
    columns=columns_to_remove,
    errors="ignore"
)

# Merge cleanly
scoring_game_data = scoring_game_data.merge(
    season_scoring_environment,
    on="season",
    how="left"
)

print(
    scoring_game_data[
        [
            "season",
            "prior_league_game_total"
        ]
    ].drop_duplicates().sort_values("season")
)

      season  prior_league_game_total
0       2019                46.687500
240     2020                45.625000
496     2021                49.578125
768     2022                45.963235
1039    2023                43.763838
1311    2024                43.536765
1583    2025                45.823529


In [223]:
scoring_game_data["raw_matchup_adjustment"] = (
    scoring_game_data["projected_total"]
    - scoring_game_data["prior_league_game_total"]
)

In [224]:
shrinkage_results = []

for weight in np.arange(0.0, 1.05, 0.05):

    season_results = []

    for season in range(2019, 2026):

        test = scoring_game_data[
            scoring_game_data["season"] == season
        ].copy()

        test["shrunk_projected_total"] = (
            test["prior_league_game_total"]
            + weight * test["raw_matchup_adjustment"]
        )

        mae = mean_absolute_error(
            test["total_points"],
            test["shrunk_projected_total"]
        )

        season_results.append(mae)

    shrinkage_results.append(
        {
            "team_influence_weight": weight,
            "average_mae": np.mean(season_results)
        }
    )

shrinkage_results = pd.DataFrame(
    shrinkage_results
)

display(
    shrinkage_results.round(3)
)

,team_influence_weight,average_mae
0,0.00,11.052
1,0.05,11.047
2,0.10,11.042
3,0.15,11.038
4,0.20,11.034
5,0.25,11.031
6,0.30,11.028
7,0.35,11.026
8,0.40,11.024
9,0.45,11.022


In [225]:
best_shrinkage = (
    shrinkage_results
    .sort_values("average_mae")
    .iloc[0]
)

BEST_TOTAL_WEIGHT = (
    best_shrinkage["team_influence_weight"]
)

print(
    f"Best Team Influence Weight: "
    f"{BEST_TOTAL_WEIGHT:.2f}"
)

print(
    f"Best Historical MAE: "
    f"{best_shrinkage['average_mae']:.2f}"
)

print(
    f"Full Matchup Model MAE: "
    f"{scoring_game_validation['model_mae'].mean():.2f}"
)

Best Team Influence Weight: 0.60
Best Historical MAE: 11.02
Full Matchup Model MAE: 11.04


# 2026 League Scoring Environment

Historical testing showed that preseason team level efficiency measures did not consistently improve game total predictions over a league scoring baseline.

For Week 1, the model therefore uses the recent NFL scoring environment as the baseline expected total. The existing preseason game model remains responsible for determining the expected scoring margin and game winner.

In [226]:
league_totals_by_season = (
    historical_schedule
    .groupby("season")
    .agg(
        games=("game_id", "count"),
        average_total=("total_points", "mean")
    )
    .reset_index()
)

display(
    league_totals_by_season.round(2)
)

,season,games,average_total
0,2016,256,45.55
1,2017,256,43.44
2,2018,256,46.69
3,2019,256,45.62
4,2020,256,49.58
5,2021,272,45.96
6,2022,271,43.76
7,2023,272,43.54
8,2024,272,45.82
9,2025,272,46.03


In [227]:
recent_league_totals = (
    league_totals_by_season[
        league_totals_by_season["season"].isin(
            [2023, 2024, 2025]
        )
    ]
    .set_index("season")["average_total"]
)

PROJECTED_2026_LEAGUE_TOTAL = (
    0.60 * recent_league_totals.loc[2025]
    + 0.25 * recent_league_totals.loc[2024]
    + 0.15 * recent_league_totals.loc[2023]
)

print(
    f"Projected 2026 League Scoring Baseline: "
    f"{PROJECTED_2026_LEAGUE_TOTAL:.2f} points/game"
)

Projected 2026 League Scoring Baseline: 45.60 points/game


In [228]:
week_1["projected_total"] = (
    PROJECTED_2026_LEAGUE_TOTAL
)

week_1["projected_home_score"] = (
    week_1["projected_total"]
    + week_1["expected_home_margin"]
) / 2

week_1["projected_away_score"] = (
    week_1["projected_total"]
    - week_1["expected_home_margin"]
) / 2

display(
    week_1[
        [
            "away_team",
            "home_team",
            "projected_away_score",
            "projected_home_score",
            "projected_total",
            "expected_home_margin",
            "predicted_winner"
        ]
    ].round(2)
)

,away_team,home_team,projected_away_score,projected_home_score,projected_total,expected_home_margin,predicted_winner
0,NE,SEA,20.69,24.91,45.6,4.22,SEA
1,SF,LA,21.60,24.00,45.6,2.40,LA
2,ARI,LAC,19.62,25.98,45.6,6.35,LAC
3,ATL,PIT,20.98,24.62,45.6,3.64,PIT
4,BAL,IND,23.19,22.41,45.6,-0.78,BAL
5,BUF,HOU,22.92,22.68,45.6,-0.24,BUF
6,CHI,CAR,24.16,21.44,45.6,-2.72,CHI
7,CLE,JAX,18.99,26.61,45.6,7.63,JAX
8,DAL,NYG,23.45,22.15,45.6,-1.31,DAL
9,GB,MIN,22.95,22.65,45.6,-0.30,GB


# Week 1 Betting Market

The model's projected margin and total are independent of sportsbook lines.

Market spreads and totals are added only after the projections are generated so that the model can be evaluated against the betting market and used to identify potential ATS and over/under edges.

In [229]:
schedule_2026 = nfl.load_schedules(
    seasons=[2026]
).to_pandas()

print("Schedule Shape:", schedule_2026.shape)

market_columns = [
    col for col in schedule_2026.columns
    if any(
        term in col.lower()
        for term in [
            "spread",
            "line",
            "total",
            "money"
        ]
    )
]

print("\nPotential Market Columns:")
print(market_columns)

Schedule Shape: (272, 46)

Potential Market Columns:
['total', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line']


In [230]:
week_1_market = (
    schedule_2026[
        schedule_2026["week"] == 1
    ]
    [
        [
            "game_id",
            "away_team",
            "home_team",
            "spread_line",
            "away_spread_odds",
            "home_spread_odds",
            "total_line",
            "away_moneyline",
            "home_moneyline"
        ]
    ]
    .copy()
)

display(week_1_market)

,game_id,away_team,home_team,spread_line,away_spread_odds,home_spread_odds,total_line,away_moneyline,home_moneyline
0,2026_01_NE_SEA,NE,SEA,3.0,-102.0,-118.0,44.5,140.0,-166.0
1,2026_01_SF_LA,SF,LA,3.5,-102.0,-118.0,48.5,164.0,-198.0
2,2026_01_CHI_CAR,CHI,CAR,-3.0,-118.0,-102.0,46.5,-162.0,136.0
3,2026_01_TB_CIN,TB,CIN,3.5,-108.0,-112.0,50.5,164.0,-198.0
4,2026_01_NO_DET,NO,DET,7.0,-112.0,-108.0,50.5,250.0,-310.0
5,2026_01_BUF_HOU,BUF,HOU,-1.5,-108.0,-112.0,44.5,-120.0,100.0
6,2026_01_BAL_IND,BAL,IND,-3.5,-105.0,-115.0,47.5,-175.0,145.0
7,2026_01_CLE_JAX,CLE,JAX,8.5,-110.0,-110.0,40.5,350.0,-455.0
8,2026_01_ATL_PIT,ATL,PIT,3.5,-108.0,-112.0,42.5,154.0,-185.0
9,2026_01_NYJ_TEN,NYJ,TEN,1.5,-110.0,-110.0,38.5,105.0,-125.0


In [231]:
# Convert nflverse spread convention into standard sportsbook notation.
#
# nflverse:
#   +3.0 = home team favored by 3
#   -3.0 = away team favored by 3
#
# Standard betting notation:
#   favorite = negative spread
#   underdog = positive spread

week_1_market["home_spread"] = (
    -week_1_market["spread_line"]
)

week_1_market["away_spread"] = (
    week_1_market["spread_line"]
)

week_1_market["market_favorite"] = np.where(
    week_1_market["home_spread"] < 0,
    week_1_market["home_team"],
    np.where(
        week_1_market["away_spread"] < 0,
        week_1_market["away_team"],
        "PICK"
    )
)

display(
    week_1_market[
        [
            "away_team",
            "home_team",
            "away_spread",
            "home_spread",
            "market_favorite",
            "total_line"
        ]
    ]
)

,away_team,home_team,away_spread,home_spread,market_favorite,total_line
0,NE,SEA,3.0,-3.0,SEA,44.5
1,SF,LA,3.5,-3.5,LA,48.5
2,CHI,CAR,-3.0,3.0,CHI,46.5
3,TB,CIN,3.5,-3.5,CIN,50.5
4,NO,DET,7.0,-7.0,DET,50.5
5,BUF,HOU,-1.5,1.5,BUF,44.5
6,BAL,IND,-3.5,3.5,BAL,47.5
7,CLE,JAX,8.5,-8.5,JAX,40.5
8,ATL,PIT,3.5,-3.5,PIT,42.5
9,NYJ,TEN,1.5,-1.5,TEN,38.5


# Week 1 ATS & Total Projections

Sportsbook spreads and totals are compared with the model only after the model projections have been generated.

All spreads are displayed using standard betting notation:

- Negative spread = favorite
- Positive spread = underdog

For spreads, the model's expected scoring margin is converted into a model implied betting spread and compared with the sportsbook line.

For totals, the model's projected scoring environment is compared with the sportsbook total.

These market lines are used only as comparison benchmarks and do not influence the underlying model predictions.

In [232]:
week_1 = week_1.merge(
    week_1_market[
        [
            "game_id",
            "away_spread",
            "home_spread",
            "market_favorite",
            "away_spread_odds",
            "home_spread_odds",
            "total_line",
            "away_moneyline",
            "home_moneyline"
        ]
    ],
    on="game_id",
    how="left"
)

print("Week 1 Games:", len(week_1))

print(
    "Games With Spread:",
    week_1["home_spread"].notna().sum()
)

print(
    "Games With Total:",
    week_1["total_line"].notna().sum()
)

Week 1 Games: 16
Games With Spread: 16
Games With Total: 16


In [233]:
week_1["model_home_spread"] = (
    -week_1["expected_home_margin"]
)

week_1["model_away_spread"] = (
    week_1["expected_home_margin"]
)

In [234]:
week_1["home_ats_edge"] = (
    week_1["home_spread"]
    - week_1["model_home_spread"]
)

week_1["away_ats_edge"] = (
    week_1["away_spread"]
    - week_1["model_away_spread"]
)

week_1["ats_pick"] = np.where(
    week_1["home_ats_edge"] > 0,
    week_1["home_team"],
    np.where(
        week_1["away_ats_edge"] > 0,
        week_1["away_team"],
        "PUSH"
    )
)

week_1["ats_edge_size"] = (
    week_1[
        [
            "home_ats_edge",
            "away_ats_edge"
        ]
    ]
    .max(axis=1)
)

In [235]:
def format_ats_pick(row):

    if row["ats_pick"] == "PUSH":
        return "PUSH"

    if row["ats_pick"] == row["home_team"]:
        line = row["home_spread"]
    else:
        line = row["away_spread"]

    if line > 0:
        return f"{row['ats_pick']} +{line:.1f}"

    return f"{row['ats_pick']} {line:.1f}"


week_1["ats_pick_display"] = (
    week_1.apply(
        format_ats_pick,
        axis=1
    )
)

In [236]:
week_1["total_edge"] = (
    week_1["projected_total"]
    - week_1["total_line"]
)

week_1["total_pick"] = np.where(
    week_1["total_edge"] > 0,
    "OVER",
    np.where(
        week_1["total_edge"] < 0,
        "UNDER",
        "PUSH"
    )
)

week_1["total_edge_size"] = (
    week_1["total_edge"].abs()
)

In [237]:
display(
    week_1[
        [
            "away_team",
            "home_team",
            "projected_away_score",
            "projected_home_score",
            "predicted_winner",
            "predicted_win_probability",
            "model_away_spread",
            "model_home_spread",
            "away_spread",
            "home_spread",
            "ats_pick_display",
            "ats_edge_size",
            "projected_total",
            "total_line",
            "total_pick",
            "total_edge_size"
        ]
    ]
    .round(2)
)

,away_team,home_team,projected_away_score,projected_home_score,predicted_winner,predicted_win_probability,model_away_spread,model_home_spread,away_spread,home_spread,ats_pick_display,ats_edge_size,projected_total,total_line,total_pick,total_edge_size
0,NE,SEA,20.69,24.91,SEA,0.62,4.22,-4.22,3.0,-3.0,SEA -3.0,1.22,45.6,44.5,OVER,1.1
1,SF,LA,21.60,24.00,LA,0.57,2.40,-2.40,3.5,-3.5,SF +3.5,1.10,45.6,48.5,UNDER,2.9
2,ARI,LAC,19.62,25.98,LAC,0.67,6.35,-6.35,9.5,-9.5,ARI +9.5,3.15,45.6,47.5,UNDER,1.9
3,ATL,PIT,20.98,24.62,PIT,0.60,3.64,-3.64,3.5,-3.5,PIT -3.5,0.14,45.6,42.5,OVER,3.1
4,BAL,IND,23.19,22.41,BAL,0.52,-0.78,0.78,-3.5,3.5,IND +3.5,2.72,45.6,47.5,UNDER,1.9
5,BUF,HOU,22.92,22.68,BUF,0.51,-0.24,0.24,-1.5,1.5,HOU +1.5,1.26,45.6,44.5,OVER,1.1
6,CHI,CAR,24.16,21.44,CHI,0.58,-2.72,2.72,-3.0,3.0,CAR +3.0,0.28,45.6,46.5,UNDER,0.9
7,CLE,JAX,18.99,26.61,JAX,0.70,7.63,-7.63,8.5,-8.5,CLE +8.5,0.87,45.6,40.5,OVER,5.1
8,DAL,NYG,23.45,22.15,DAL,0.54,-1.31,1.31,-3.0,3.0,NYG +3.0,1.69,45.6,48.5,UNDER,2.9
9,GB,MIN,22.95,22.65,GB,0.51,-0.30,0.30,1.5,-1.5,GB +1.5,1.80,45.6,46.5,UNDER,0.9


# Edge Strength Classification

Not every disagreement between the model and the betting market should be treated equally.

Small differences may simply reflect normal model uncertainty, while larger differences represent more meaningful disagreements.

Historical validation will eventually be used to determine the most appropriate betting thresholds. For the initial Week 1 output, edge sizes are grouped into simple descriptive categories so that weak disagreements are not presented with the same confidence as larger ones.

In [238]:
def classify_ats_edge(edge):

    if edge < 1.0:
        return "No Edge"

    elif edge < 2.0:
        return "Lean"

    elif edge < 3.0:
        return "Moderate Edge"

    else:
        return "Strong Edge"


week_1["ats_edge_class"] = (
    week_1["ats_edge_size"]
    .apply(classify_ats_edge)
)

In [239]:
def classify_total_edge(edge):

    if edge < 2.0:
        return "No Edge"

    elif edge < 3.5:
        return "Lean"

    elif edge < 5.0:
        return "Moderate Edge"

    else:
        return "Strong Edge"


week_1["total_edge_class"] = (
    week_1["total_edge_size"]
    .apply(classify_total_edge)
)

In [240]:
week_1_board = (
    week_1[
        [
            "away_team",
            "home_team",
            "projected_away_score",
            "projected_home_score",
            "predicted_winner",
            "predicted_win_probability",
            "ats_pick_display",
            "ats_edge_size",
            "ats_edge_class",
            "projected_total",
            "total_line",
            "total_pick",
            "total_edge_size",
            "total_edge_class"
        ]
    ]
    .copy()
)

display(
    week_1_board.round(2)
)

,away_team,home_team,projected_away_score,projected_home_score,predicted_winner,predicted_win_probability,ats_pick_display,ats_edge_size,ats_edge_class,projected_total,total_line,total_pick,total_edge_size,total_edge_class
0,NE,SEA,20.69,24.91,SEA,0.62,SEA -3.0,1.22,Lean,45.6,44.5,OVER,1.1,No Edge
1,SF,LA,21.60,24.00,LA,0.57,SF +3.5,1.10,Lean,45.6,48.5,UNDER,2.9,Lean
2,ARI,LAC,19.62,25.98,LAC,0.67,ARI +9.5,3.15,Strong Edge,45.6,47.5,UNDER,1.9,No Edge
3,ATL,PIT,20.98,24.62,PIT,0.60,PIT -3.5,0.14,No Edge,45.6,42.5,OVER,3.1,Lean
4,BAL,IND,23.19,22.41,BAL,0.52,IND +3.5,2.72,Moderate Edge,45.6,47.5,UNDER,1.9,No Edge
5,BUF,HOU,22.92,22.68,BUF,0.51,HOU +1.5,1.26,Lean,45.6,44.5,OVER,1.1,No Edge
6,CHI,CAR,24.16,21.44,CHI,0.58,CAR +3.0,0.28,No Edge,45.6,46.5,UNDER,0.9,No Edge
7,CLE,JAX,18.99,26.61,JAX,0.70,CLE +8.5,0.87,No Edge,45.6,40.5,OVER,5.1,Strong Edge
8,DAL,NYG,23.45,22.15,DAL,0.54,NYG +3.0,1.69,Lean,45.6,48.5,UNDER,2.9,Lean
9,GB,MIN,22.95,22.65,GB,0.51,GB +1.5,1.80,Lean,45.6,46.5,UNDER,0.9,No Edge


# Historical ATS Validation

The Week 1 model can identify games where its expected margin differs from the sportsbook spread, but a larger disagreement does not automatically mean a better betting opportunity.

To evaluate whether these differences are meaningful, historical seasons are reconstructed using only information that would have been available before each season.

The goal is to test whether larger model versus market spread differences historically produced stronger against-the-spread results.

In [241]:
historical_strength = (
    historical_team_features[
        [
            "season",
            "team",
            "point_diff_per_game"
        ]
    ]
    .copy()
    .sort_values(["team", "season"])
)

for lag in [1, 2, 3]:

    historical_strength[
        f"pd_lag{lag}"
    ] = (
        historical_strength
        .groupby("team")["point_diff_per_game"]
        .shift(lag)
    )

historical_strength["weighted_prior_pd"] = (
    0.60 * historical_strength["pd_lag1"]
    + 0.25 * historical_strength["pd_lag2"]
    + 0.15 * historical_strength["pd_lag3"]
)

historical_strength_model_data = (
    historical_strength
    .dropna(
        subset=[
            "weighted_prior_pd",
            "point_diff_per_game"
        ]
    )
    .reset_index(drop=True)
)

In [242]:
historical_preseason_strengths = []

for test_season in range(2019, 2026):

    train = historical_strength_model_data[
        historical_strength_model_data["season"] < test_season
    ].copy()

    test = historical_strength_model_data[
        historical_strength_model_data["season"] == test_season
    ].copy()

    strength_model = LinearRegression()

    strength_model.fit(
        train[["weighted_prior_pd"]],
        train["point_diff_per_game"]
    )

    test["predicted_team_strength"] = (
        strength_model.predict(
            test[["weighted_prior_pd"]]
        )
    )

    historical_preseason_strengths.append(
        test[
            [
                "season",
                "team",
                "predicted_team_strength"
            ]
        ]
    )

historical_preseason_strengths = pd.concat(
    historical_preseason_strengths,
    ignore_index=True
)

display(
    historical_preseason_strengths.head(10).round(3)
)

,season,team,predicted_team_strength
0,2019,ARI,-4.407
1,2019,ATL,0.832
2,2019,BAL,3.010
3,2019,BUF,-2.549
4,2019,CAR,0.015
5,2019,CHI,1.747
6,2019,CIN,-2.251
7,2019,CLE,-3.165
8,2019,DAL,1.092
9,2019,DEN,-1.027


In [243]:
home_strength = (
    historical_preseason_strengths.rename(
        columns={
            "team": "home_team",
            "predicted_team_strength": "home_team_strength"
        }
    )
)

away_strength = (
    historical_preseason_strengths.rename(
        columns={
            "team": "away_team",
            "predicted_team_strength": "away_team_strength"
        }
    )
)

historical_ats = (
    historical_schedule
    .merge(
        home_strength,
        on=["season", "home_team"],
        how="inner"
    )
    .merge(
        away_strength,
        on=["season", "away_team"],
        how="inner"
    )
)

historical_ats = historical_ats.dropna(
    subset=[
        "spread_line",
        "result"
    ]
).reset_index(drop=True)

print(
    "Historical ATS Games:",
    len(historical_ats)
)

Historical ATS Games: 1855


In [244]:
historical_ats["model_home_margin"] = np.nan

for test_season in range(2019, 2026):

    prior_games = historical_schedule[
        (historical_schedule["season"] < test_season)
        & (historical_schedule["location"] == "Home")
        & (historical_schedule["result"].notna())
    ].copy()

    historical_hfa = (
        prior_games["result"].mean()
    )

    mask = (
        historical_ats["season"] == test_season
    )

    historical_ats.loc[
        mask,
        "model_home_margin"
    ] = (
        historical_ats.loc[
            mask,
            "home_team_strength"
        ]
        - historical_ats.loc[
            mask,
            "away_team_strength"
        ]
        + historical_hfa
    )

In [245]:
historical_ats["home_spread"] = (
    -historical_ats["spread_line"]
)

historical_ats["away_spread"] = (
    historical_ats["spread_line"]
)

historical_ats["model_home_spread"] = (
    -historical_ats["model_home_margin"]
)

historical_ats["model_away_spread"] = (
    historical_ats["model_home_margin"]
)

In [246]:
historical_ats["home_ats_edge"] = (
    historical_ats["home_spread"]
    - historical_ats["model_home_spread"]
)

historical_ats["away_ats_edge"] = (
    historical_ats["away_spread"]
    - historical_ats["model_away_spread"]
)

historical_ats["model_ats_side"] = np.where(
    historical_ats["home_ats_edge"] > 0,
    "HOME",
    np.where(
        historical_ats["away_ats_edge"] > 0,
        "AWAY",
        "PUSH"
    )
)

historical_ats["ats_edge_size"] = (
    historical_ats[
        [
            "home_ats_edge",
            "away_ats_edge"
        ]
    ]
    .max(axis=1)
)

In [247]:
historical_ats["ats_result_margin"] = np.where(
    historical_ats["model_ats_side"] == "HOME",
    historical_ats["result"]
    + historical_ats["home_spread"],
    -historical_ats["result"]
    + historical_ats["away_spread"]
)

historical_ats["ats_result"] = np.where(
    historical_ats["ats_result_margin"] > 0,
    "WIN",
    np.where(
        historical_ats["ats_result_margin"] < 0,
        "LOSS",
        "PUSH"
    )
)

print(
    historical_ats["ats_result"]
    .value_counts()
)

ats_result
LOSS    920
WIN     892
PUSH     43
Name: count, dtype: int64


In [248]:
ats_threshold_results = []

for threshold in [
    0.0,
    0.5,
    1.0,
    1.5,
    2.0,
    2.5,
    3.0,
    3.5,
    4.0,
    5.0
]:

    sample = historical_ats[
        historical_ats["ats_edge_size"]
        >= threshold
    ].copy()

    graded = sample[
        sample["ats_result"] != "PUSH"
    ]

    wins = (
        graded["ats_result"] == "WIN"
    ).sum()

    losses = (
        graded["ats_result"] == "LOSS"
    ).sum()

    win_rate = (
        wins / (wins + losses)
        if (wins + losses) > 0
        else np.nan
    )

    ats_threshold_results.append(
        {
            "minimum_edge": threshold,
            "bets": wins + losses,
            "wins": wins,
            "losses": losses,
            "win_rate": win_rate
        }
    )

ats_threshold_results = pd.DataFrame(
    ats_threshold_results
)

display(
    ats_threshold_results.round(3)
)

,minimum_edge,bets,wins,losses,win_rate
0,0.0,1812,892,920,0.492
1,0.5,1684,825,859,0.490
2,1.0,1555,770,785,0.495
3,1.5,1396,691,705,0.495
4,2.0,1264,642,622,0.508
5,2.5,1144,584,560,0.510
6,3.0,1022,524,498,0.513
7,3.5,914,477,437,0.522
8,4.0,801,413,388,0.516
9,5.0,610,307,303,0.503


# Interpreting ATS Disagreements

Historical testing shows that larger differences between the model and sportsbook spreads contain some additional signal, but the relationship is not strong enough to treat model edge size as betting confidence.

ATS differences are therefore presented as levels of model-market disagreement rather than as recommended bets.

The strongest historical performance occurred around differences of 3.5 points or more, although performance was not consistently better as the disagreement increased further.

In [249]:
def classify_ats_disagreement(edge):

    if edge < 2.0:
        return "Small"

    elif edge < 3.5:
        return "Moderate"

    else:
        return "Large"


week_1["ats_disagreement"] = (
    week_1["ats_edge_size"]
    .apply(classify_ats_disagreement)
)

In [250]:
ats_by_season = []

for season in range(2019, 2026):

    for threshold in [2.0, 3.5]:

        sample = historical_ats[
            (historical_ats["season"] == season)
            & (historical_ats["ats_edge_size"] >= threshold)
            & (historical_ats["ats_result"] != "PUSH")
        ]

        wins = (
            sample["ats_result"] == "WIN"
        ).sum()

        losses = (
            sample["ats_result"] == "LOSS"
        ).sum()

        win_rate = (
            wins / (wins + losses)
            if (wins + losses) > 0
            else np.nan
        )

        ats_by_season.append(
            {
                "season": season,
                "minimum_edge": threshold,
                "bets": wins + losses,
                "wins": wins,
                "losses": losses,
                "win_rate": win_rate
            }
        )

ats_by_season = pd.DataFrame(
    ats_by_season
)

display(
    ats_by_season.round(3)
)

,season,minimum_edge,bets,wins,losses,win_rate
0,2019,2.0,157,82,75,0.522
1,2019,3.5,109,56,53,0.514
2,2020,2.0,194,100,94,0.515
3,2020,3.5,139,76,63,0.547
4,2021,2.0,192,107,85,0.557
5,2021,3.5,156,86,70,0.551
6,2022,2.0,183,91,92,0.497
7,2022,3.5,138,76,62,0.551
8,2023,2.0,175,88,87,0.503
9,2023,3.5,115,61,54,0.530


In [251]:
ats_season_pivot = (
    ats_by_season
    .pivot(
        index="season",
        columns="minimum_edge",
        values="win_rate"
    )
    .rename(
        columns={
            2.0: "edge_2_plus",
            3.5: "edge_3_5_plus"
        }
    )
)

display(
    ats_season_pivot.round(3)
)

minimum_edge,edge_2_plus,edge_3_5_plus
season,,
2019,0.522,0.514
2020,0.515,0.547
2021,0.557,0.551
2022,0.497,0.551
2023,0.503,0.530
2024,0.487,0.483
2025,0.471,0.465


# ATS Validation Conclusion

Historical testing did not show a sufficiently stable relationship between model-market spread differences and ATS win rate.

Larger disagreements performed better during several seasons, but the pattern was inconsistent and weakened considerably in 2024 and 2025.

For that reason, ATS projections are presented as model versus market comparisons rather than confidence rated betting recommendations. The size of the disagreement is retained as useful context, but no categorical betting strength label is assigned.

In [252]:
week_1 = week_1.drop(
    columns=[
        "ats_edge_class",
        "ats_disagreement"
    ],
    errors="ignore"
)

In [253]:
week_1 = week_1.rename(
    columns={
        "ats_edge_size": "ats_difference"
    }
)

# Historical Over/Under Validation

The Week 1 total points projection uses the recent NFL scoring environment rather than a matchup specific totals model because historical testing showed that preseason team level efficiency measures did not consistently improve game total predictions.

To evaluate this approach, the same 60/25/15 league scoring baseline is reconstructed historically using only information available before each season.

The projected league total is then compared with each game's sportsbook total to evaluate whether larger model-market differences historically provided useful over/under information.

In [254]:
historical_total_baselines = []

league_total_lookup = (
    league_totals_by_season
    .set_index("season")["average_total"]
)

for season in range(2019, 2026):

    projected_league_total = (
        0.60 * league_total_lookup.loc[season - 1]
        + 0.25 * league_total_lookup.loc[season - 2]
        + 0.15 * league_total_lookup.loc[season - 3]
    )

    historical_total_baselines.append(
        {
            "season": season,
            "projected_league_total": projected_league_total
        }
    )

historical_total_baselines = pd.DataFrame(
    historical_total_baselines
)

display(
    historical_total_baselines.round(2)
)

,season,projected_league_total
0,2019,45.70
1,2020,45.56
2,2021,48.16
3,2022,46.82
4,2023,45.19
5,2024,43.96
6,2025,44.94


In [255]:
historical_ou = (
    historical_schedule
    .merge(
        historical_total_baselines,
        on="season",
        how="inner"
    )
    .dropna(
        subset=[
            "total_line",
            "total_points"
        ]
    )
    .copy()
)

historical_ou["total_difference"] = (
    historical_ou["projected_league_total"]
    - historical_ou["total_line"]
)

historical_ou["ou_pick"] = np.where(
    historical_ou["total_difference"] > 0,
    "OVER",
    np.where(
        historical_ou["total_difference"] < 0,
        "UNDER",
        "PUSH"
    )
)

historical_ou["total_difference_size"] = (
    historical_ou["total_difference"].abs()
)

print(
    "Historical O/U Games:",
    len(historical_ou)
)

Historical O/U Games: 1871


In [256]:
historical_ou["actual_ou_margin"] = (
    historical_ou["total_points"]
    - historical_ou["total_line"]
)

historical_ou["ou_result"] = np.where(
    historical_ou["actual_ou_margin"] == 0,
    "PUSH",
    np.where(
        (
            (historical_ou["ou_pick"] == "OVER")
            & (historical_ou["actual_ou_margin"] > 0)
        )
        |
        (
            (historical_ou["ou_pick"] == "UNDER")
            & (historical_ou["actual_ou_margin"] < 0)
        ),
        "WIN",
        "LOSS"
    )
)

print(
    historical_ou["ou_result"]
    .value_counts()
)

ou_result
LOSS    958
WIN     896
PUSH     17
Name: count, dtype: int64


In [257]:
ou_threshold_results = []

for threshold in [
    0.0,
    0.5,
    1.0,
    1.5,
    2.0,
    2.5,
    3.0,
    3.5,
    4.0,
    5.0,
    6.0
]:

    sample = historical_ou[
        (
            historical_ou["total_difference_size"]
            >= threshold
        )
        & (
            historical_ou["ou_result"] != "PUSH"
        )
    ]

    wins = (
        sample["ou_result"] == "WIN"
    ).sum()

    losses = (
        sample["ou_result"] == "LOSS"
    ).sum()

    win_rate = (
        wins / (wins + losses)
        if wins + losses > 0
        else np.nan
    )

    ou_threshold_results.append(
        {
            "minimum_difference": threshold,
            "bets": wins + losses,
            "wins": wins,
            "losses": losses,
            "win_rate": win_rate
        }
    )

ou_threshold_results = pd.DataFrame(
    ou_threshold_results
)

display(
    ou_threshold_results.round(3)
)

,minimum_difference,bets,wins,losses,win_rate
0,0.0,1854,896,958,0.483
1,0.5,1682,820,862,0.488
2,1.0,1542,752,790,0.488
3,1.5,1382,675,707,0.488
4,2.0,1246,611,635,0.490
5,2.5,1102,548,554,0.497
6,3.0,939,465,474,0.495
7,3.5,813,408,405,0.502
8,4.0,686,349,337,0.509
9,5.0,509,256,253,0.503


# Over/Under Validation Conclusion

Historical testing did not show that the preseason league scoring baseline produced a reliable betting advantage against sportsbook totals.

Larger differences between the projected scoring environment and the market total showed modest improvement in some ranges, but the relationship was not strong or consistent enough to interpret the difference as betting confidence.

The model will still produce an over/under projection for each game, but these should be interpreted as model versus market comparisons rather than validated betting recommendations.

In [258]:
week_1 = week_1.drop(
    columns=[
        "total_edge_class"
    ],
    errors="ignore"
)

week_1 = week_1.rename(
    columns={
        "total_edge": "total_difference_signed",
        "total_edge_size": "total_difference"
    }
)

In [259]:
week_1["projected_away_score_final"] = (
    np.floor(
        week_1["projected_away_score"] + 0.5
    )
    .astype(int)
)

week_1["projected_home_score_final"] = (
    np.floor(
        week_1["projected_home_score"] + 0.5
    )
    .astype(int)
)

week_1["projected_score"] = (
    week_1["away_team"]
    + " "
    + week_1["projected_away_score_final"].astype(str)
    + " - "
    + week_1["home_team"]
    + " "
    + week_1["projected_home_score_final"].astype(str)
)

In [260]:
def format_market_spread(row):

    if row["home_spread"] < 0:
        return (
            f"{row['home_team']} "
            f"{row['home_spread']:.1f}"
        )

    elif row["away_spread"] < 0:
        return (
            f"{row['away_team']} "
            f"{row['away_spread']:.1f}"
        )

    return "PICK"


week_1["market_spread_display"] = (
    week_1.apply(
        format_market_spread,
        axis=1
    )
)

In [261]:
def format_model_spread(row):

    if row["model_home_spread"] < 0:
        return (
            f"{row['home_team']} "
            f"{row['model_home_spread']:.1f}"
        )

    elif row["model_away_spread"] < 0:
        return (
            f"{row['away_team']} "
            f"{row['model_away_spread']:.1f}"
        )

    return "PICK"


week_1["model_spread_display"] = (
    week_1.apply(
        format_model_spread,
        axis=1
    )
)

In [262]:
week_1_final = (
    week_1[
        [
            "away_team",
            "home_team",
            "projected_score",
            "predicted_winner",
            "predicted_win_probability",
            "model_spread_display",
            "market_spread_display",
            "ats_pick_display",
            "ats_difference",
            "projected_total",
            "total_line",
            "total_pick",
            "total_difference"
        ]
    ]
    .copy()
)

week_1_final = week_1_final.rename(
    columns={
        "away_team": "Away",
        "home_team": "Home",
        "projected_score": "Projected Score",
        "predicted_winner": "SU Pick",
        "predicted_win_probability": "Win Probability",
        "model_spread_display": "Model Spread",
        "market_spread_display": "Market Spread",
        "ats_pick_display": "ATS Pick",
        "ats_difference": "ATS Difference",
        "projected_total": "Model Total",
        "total_line": "Market Total",
        "total_pick": "O/U Pick",
        "total_difference": "Total Difference"
    }
)

display(
    week_1_final.round(2)
)

,Away,Home,Projected Score,SU Pick,Win Probability,Model Spread,Market Spread,ATS Pick,ATS Difference,Model Total,Market Total,O/U Pick,Total Difference
0,NE,SEA,NE 21 - SEA 25,SEA,0.62,SEA -4.2,SEA -3.0,SEA -3.0,1.22,45.6,44.5,OVER,1.1
1,SF,LA,SF 22 - LA 24,LA,0.57,LA -2.4,LA -3.5,SF +3.5,1.10,45.6,48.5,UNDER,2.9
2,ARI,LAC,ARI 20 - LAC 26,LAC,0.67,LAC -6.4,LAC -9.5,ARI +9.5,3.15,45.6,47.5,UNDER,1.9
3,ATL,PIT,ATL 21 - PIT 25,PIT,0.60,PIT -3.6,PIT -3.5,PIT -3.5,0.14,45.6,42.5,OVER,3.1
4,BAL,IND,BAL 23 - IND 22,BAL,0.52,BAL -0.8,BAL -3.5,IND +3.5,2.72,45.6,47.5,UNDER,1.9
5,BUF,HOU,BUF 23 - HOU 23,BUF,0.51,BUF -0.2,BUF -1.5,HOU +1.5,1.26,45.6,44.5,OVER,1.1
6,CHI,CAR,CHI 24 - CAR 21,CHI,0.58,CHI -2.7,CHI -3.0,CAR +3.0,0.28,45.6,46.5,UNDER,0.9
7,CLE,JAX,CLE 19 - JAX 27,JAX,0.70,JAX -7.6,JAX -8.5,CLE +8.5,0.87,45.6,40.5,OVER,5.1
8,DAL,NYG,DAL 23 - NYG 22,DAL,0.54,DAL -1.3,DAL -3.0,NYG +3.0,1.69,45.6,48.5,UNDER,2.9
9,GB,MIN,GB 23 - MIN 23,GB,0.51,GB -0.3,MIN -1.5,GB +1.5,1.80,45.6,46.5,UNDER,0.9


# Final Week 1 Output

The final Week 1 projection board combines the preseason model's game predictions with sportsbook spreads and totals.

Projected scores retain one decimal place so that close games remain consistent with the model's underlying expected margin rather than being converted into artificial ties through rounding.

Market lines are used only as comparison benchmarks and do not influence the underlying game projections.

In [263]:
week_1["projected_score"] = (
    week_1["away_team"]
    + " "
    + week_1["projected_away_score"].round(1).astype(str)
    + " - "
    + week_1["home_team"]
    + " "
    + week_1["projected_home_score"].round(1).astype(str)
)

In [264]:
week_1_final = (
    week_1[
        [
            "away_team",
            "home_team",
            "projected_score",
            "predicted_winner",
            "predicted_win_probability",
            "model_spread_display",
            "market_spread_display",
            "ats_pick_display",
            "ats_difference",
            "projected_total",
            "total_line",
            "total_pick",
            "total_difference"
        ]
    ]
    .copy()
)

week_1_final = week_1_final.rename(
    columns={
        "away_team": "Away",
        "home_team": "Home",
        "projected_score": "Projected Score",
        "predicted_winner": "SU Pick",
        "predicted_win_probability": "Win Probability",
        "model_spread_display": "Model Spread",
        "market_spread_display": "Market Spread",
        "ats_pick_display": "ATS Pick",
        "ats_difference": "ATS Difference",
        "projected_total": "Model Total",
        "total_line": "Market Total",
        "total_pick": "O/U Pick",
        "total_difference": "Total Difference"
    }
)

display(
    week_1_final.round(2)
)

,Away,Home,Projected Score,SU Pick,Win Probability,Model Spread,Market Spread,ATS Pick,ATS Difference,Model Total,Market Total,O/U Pick,Total Difference
0,NE,SEA,NE 20.7 - SEA 24.9,SEA,0.62,SEA -4.2,SEA -3.0,SEA -3.0,1.22,45.6,44.5,OVER,1.1
1,SF,LA,SF 21.6 - LA 24.0,LA,0.57,LA -2.4,LA -3.5,SF +3.5,1.10,45.6,48.5,UNDER,2.9
2,ARI,LAC,ARI 19.6 - LAC 26.0,LAC,0.67,LAC -6.4,LAC -9.5,ARI +9.5,3.15,45.6,47.5,UNDER,1.9
3,ATL,PIT,ATL 21.0 - PIT 24.6,PIT,0.60,PIT -3.6,PIT -3.5,PIT -3.5,0.14,45.6,42.5,OVER,3.1
4,BAL,IND,BAL 23.2 - IND 22.4,BAL,0.52,BAL -0.8,BAL -3.5,IND +3.5,2.72,45.6,47.5,UNDER,1.9
5,BUF,HOU,BUF 22.9 - HOU 22.7,BUF,0.51,BUF -0.2,BUF -1.5,HOU +1.5,1.26,45.6,44.5,OVER,1.1
6,CHI,CAR,CHI 24.2 - CAR 21.4,CHI,0.58,CHI -2.7,CHI -3.0,CAR +3.0,0.28,45.6,46.5,UNDER,0.9
7,CLE,JAX,CLE 19.0 - JAX 26.6,JAX,0.70,JAX -7.6,JAX -8.5,CLE +8.5,0.87,45.6,40.5,OVER,5.1
8,DAL,NYG,DAL 23.5 - NYG 22.1,DAL,0.54,DAL -1.3,DAL -3.0,NYG +3.0,1.69,45.6,48.5,UNDER,2.9
9,GB,MIN,GB 22.9 - MIN 22.7,GB,0.51,GB -0.3,MIN -1.5,GB +1.5,1.80,45.6,46.5,UNDER,0.9


In [265]:
week_1.to_parquet(
    WEEKLY_DATA_DIR / "week_01_projections.parquet",
    index=False
)

# Clean final projection board
week_1_final.to_csv(
    WEEKLY_OUTPUT_DIR / "week_01_projections.csv",
    index=False
)

print("Saved Week 1 projections.")
print(
    WEEKLY_DATA_DIR
    / "week_01_projections.parquet"
)
print(
    WEEKLY_OUTPUT_DIR
    / "week_01_projections.csv"
)

Saved Week 1 projections.
..\..\data\processed\weekly\week_01_projections.parquet
..\..\weekly_projections\outputs\week_01_projections.csv


# Week 1 Summary

The Week 1 projections use the completed preseason team strength and game prediction models as the foundation for all straight up and spread predictions.

A recent league scoring baseline is used to estimate total points because historical testing did not show that the available preseason team level scoring features consistently improved game-total accuracy.

Historical testing also showed that model-market differences were not sufficiently stable to treat them as validated betting edges. ATS and over/under outputs are therefore presented as model versus market projections rather than betting recommendations.

Beginning in Week 2, actual 2026 performance can gradually be incorporated into the weekly projection process while retaining the preseason ratings as an early season prior.

# 🏈 2026 NFL Week 1 Projection Board

Straight up, spread, total, and projected score predictions from the 2026 NFL Projection Model.

**Market lines are shown for comparison only and are not inputs to the model.**

In [266]:
week_1_display = week_1_final.copy()

week_1_display["Matchup"] = (
    week_1_display["Away"]
    + " @ "
    + week_1_display["Home"]
)

week_1_display["Win Prob"] = (
    week_1_display["Win Probability"]
)

week_1_display["ATS Diff"] = (
    week_1_display["ATS Difference"]
)

week_1_display["Total Diff"] = (
    week_1_display["Total Difference"]
)

week_1_display = week_1_display[
    [
        "Matchup",
        "Projected Score",
        "SU Pick",
        "Win Prob",
        "Model Spread",
        "Market Spread",
        "ATS Pick",
        "ATS Diff",
        "Model Total",
        "Market Total",
        "O/U Pick",
        "Total Diff"
    ]
]

In [267]:
def highlight_ats_difference(value):

    if value >= 3.5:
        return "background-color: #5c2b29; color: #ffdad6; font-weight: bold;"

    elif value >= 2.0:
        return "background-color: #594600; color: #ffe088; font-weight: bold;"

    return ""


def highlight_total_difference(value):

    if value >= 5.0:
        return "background-color: #5c2b29; color: #ffdad6; font-weight: bold;"

    elif value >= 3.0:
        return "background-color: #594600; color: #ffe088; font-weight: bold;"

    return ""


def highlight_ou_pick(value):

    if value == "OVER":
        return "color: #7ee787; font-weight: bold;"

    elif value == "UNDER":
        return "color: #79c0ff; font-weight: bold;"

    return ""


styled_week_1 = (
    week_1_display.style

    .format(
        {
            "Win Prob": "{:.1%}",
            "ATS Diff": "{:.2f}",
            "Model Total": "{:.1f}",
            "Market Total": "{:.1f}",
            "Total Diff": "{:.1f}"
        }
    )

    .map(
        highlight_ats_difference,
        subset=["ATS Diff"]
    )

    .map(
        highlight_total_difference,
        subset=["Total Diff"]
    )

    .map(
        highlight_ou_pick,
        subset=["O/U Pick"]
    )

    .set_properties(
        **{
            "text-align": "center",
            "padding": "8px 10px",
            "border-bottom": "1px solid #30363d"
        }
    )

    .set_properties(
        subset=[
            "Matchup",
            "Projected Score"
        ],
        **{
            "text-align": "left",
            "font-weight": "bold"
        }
    )

    .set_properties(
        subset=[
            "SU Pick",
            "ATS Pick"
        ],
        **{
            "font-weight": "bold"
        }
    )

    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#161b22"),
                    ("color", "#f0f6fc"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("padding", "10px"),
                    ("border-bottom", "2px solid #58a6ff")
                ]
            },

            {
                "selector": "td",
                "props": [
                    ("background-color", "#0d1117"),
                    ("color", "#c9d1d9")
                ]
            },

            {
                "selector": "tbody tr:hover td",
                "props": [
                    ("background-color", "#161b22")
                ]
            },

            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("font-family", "Arial, sans-serif"),
                    ("font-size", "14px"),
                    ("width", "100%")
                ]
            }
        ]
    )

    .hide(axis="index")
)

display(styled_week_1)

Matchup,Projected Score,SU Pick,Win Prob,Model Spread,Market Spread,ATS Pick,ATS Diff,Model Total,Market Total,O/U Pick,Total Diff
NE @ SEA,NE 20.7 - SEA 24.9,SEA,61.7%,SEA -4.2,SEA -3.0,SEA -3.0,1.22,45.6,44.5,OVER,1.1
SF @ LA,SF 21.6 - LA 24.0,LA,56.7%,LA -2.4,LA -3.5,SF +3.5,1.10,45.6,48.5,UNDER,2.9
ARI @ LAC,ARI 19.6 - LAC 26.0,LAC,67.3%,LAC -6.4,LAC -9.5,ARI +9.5,3.15,45.6,47.5,UNDER,1.9
ATL @ PIT,ATL 21.0 - PIT 24.6,PIT,60.1%,PIT -3.6,PIT -3.5,PIT -3.5,0.14,45.6,42.5,OVER,3.1
BAL @ IND,BAL 23.2 - IND 22.4,BAL,52.2%,BAL -0.8,BAL -3.5,IND +3.5,2.72,45.6,47.5,UNDER,1.9
BUF @ HOU,BUF 22.9 - HOU 22.7,BUF,50.7%,BUF -0.2,BUF -1.5,HOU +1.5,1.26,45.6,44.5,OVER,1.1
CHI @ CAR,CHI 24.2 - CAR 21.4,CHI,57.6%,CHI -2.7,CHI -3.0,CAR +3.0,0.28,45.6,46.5,UNDER,0.9
CLE @ JAX,CLE 19.0 - JAX 26.6,JAX,70.4%,JAX -7.6,JAX -8.5,CLE +8.5,0.87,45.6,40.5,OVER,5.1
DAL @ NYG,DAL 23.5 - NYG 22.1,DAL,53.7%,DAL -1.3,DAL -3.0,NYG +3.0,1.69,45.6,48.5,UNDER,2.9
GB @ MIN,GB 22.9 - MIN 22.7,GB,50.8%,GB -0.3,MIN -1.5,GB +1.5,1.80,45.6,46.5,UNDER,0.9


## 👀 Biggest Model vs. Market Disagreements

These are the Week 1 games where the model differs most from the sportsbook spread.

A larger difference does **not** imply a validated betting advantage; it simply highlights where the model and market view the matchup differently.

In [268]:
biggest_ats_differences = (
    week_1_display[
        [
            "Matchup",
            "Projected Score",
            "Model Spread",
            "Market Spread",
            "ATS Pick",
            "ATS Diff"
        ]
    ]
    .sort_values(
        "ATS Diff",
        ascending=False
    )
    .head(5)
)

display(
    biggest_ats_differences.style

    .format(
        {
            "ATS Diff": "{:.2f} pts"
        }
    )

    .set_properties(
        **{
            "text-align": "center",
            "padding": "8px 12px"
        }
    )

    .set_properties(
        subset=[
            "Matchup",
            "Projected Score"
        ],
        **{
            "text-align": "left",
            "font-weight": "bold"
        }
    )

    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#161b22"),
                    ("color", "#f0f6fc"),
                    ("font-weight", "bold"),
                    ("border-bottom", "2px solid #f85149")
                ]
            },

            {
                "selector": "td",
                "props": [
                    ("background-color", "#0d1117"),
                    ("color", "#c9d1d9")
                ]
            }
        ]
    )

    .hide(axis="index")
)

Matchup,Projected Score,Model Spread,Market Spread,ATS Pick,ATS Diff
MIA @ LV,MIA 23.5 - LV 22.1,MIA -1.5,LV -3.0,MIA +3.0,4.49 pts
ARI @ LAC,ARI 19.6 - LAC 26.0,LAC -6.4,LAC -9.5,ARI +9.5,3.15 pts
TB @ CIN,TB 22.6 - CIN 23.0,CIN -0.4,CIN -3.5,TB +3.5,3.08 pts
DEN @ KC,DEN 23.1 - KC 22.5,DEN -0.5,KC -2.5,DEN +2.5,3.00 pts
BAL @ IND,BAL 23.2 - IND 22.4,BAL -0.8,BAL -3.5,IND +3.5,2.72 pts


## 📊 Biggest Model vs. Market Total Differences

These are the Week 1 games where the model's projected total differs most from the sportsbook total.

A larger difference does **not** imply a validated betting advantage; it simply highlights where the model and market have the largest disagreement on expected scoring.

In [269]:
biggest_total_differences = (
    week_1_display[
        [
            "Matchup",
            "Projected Score",
            "Model Total",
            "Market Total",
            "O/U Pick",
            "Total Diff"
        ]
    ]
    .sort_values(
        "Total Diff",
        ascending=False
    )
    .head(5)
)

display(
    biggest_total_differences.style

    .format(
        {
            "Model Total": "{:.1f}",
            "Market Total": "{:.1f}",
            "Total Diff": "{:.1f} pts"
        }
    )

    .set_properties(
        **{
            "text-align": "center",
            "padding": "8px 12px"
        }
    )

    .set_properties(
        subset=[
            "Matchup",
            "Projected Score"
        ],
        **{
            "text-align": "left",
            "font-weight": "bold"
        }
    )

    .map(
        highlight_ou_pick,
        subset=["O/U Pick"]
    )

    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("background-color", "#161b22"),
                    ("color", "#f0f6fc"),
                    ("font-weight", "bold"),
                    ("border-bottom", "2px solid #f85149")
                ]
            },

            {
                "selector": "td",
                "props": [
                    ("background-color", "#0d1117"),
                    ("color", "#c9d1d9")
                ]
            }
        ]
    )

    .hide(axis="index")
)

Matchup,Projected Score,Model Total,Market Total,O/U Pick,Total Diff
NYJ @ TEN,NYJ 22.3 - TEN 23.3,45.6,38.5,OVER,7.1 pts
CLE @ JAX,CLE 19.0 - JAX 26.6,45.6,40.5,OVER,5.1 pts
MIA @ LV,MIA 23.5 - LV 22.1,45.6,40.5,OVER,5.1 pts
TB @ CIN,TB 22.6 - CIN 23.0,45.6,50.5,UNDER,4.9 pts
NO @ DET,NO 19.1 - DET 26.5,45.6,50.5,UNDER,4.9 pts


In [271]:
WEEK_NUMBER = 1

WEEK_OUTPUT_DIR = (
    WEEKLY_OUTPUT_DIR
    / f"week_{WEEK_NUMBER:02d}"
)

WEEK_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Create clean rounded versions for saved outputs
projection_board_save = week_1_display.copy()
ats_differences_save = biggest_ats_differences.copy()
total_differences_save = biggest_total_differences.copy()

# Round projection board
projection_board_save["Win Prob"] = (
    projection_board_save["Win Prob"].round(3)
)

projection_board_save["ATS Diff"] = (
    projection_board_save["ATS Diff"].round(2)
)

projection_board_save["Model Total"] = (
    projection_board_save["Model Total"].round(1)
)

projection_board_save["Market Total"] = (
    projection_board_save["Market Total"].round(1)
)

projection_board_save["Total Diff"] = (
    projection_board_save["Total Diff"].round(1)
)

# Round ATS differences table
ats_differences_save["ATS Diff"] = (
    ats_differences_save["ATS Diff"].round(2)
)

# Round total differences table
total_differences_save["Model Total"] = (
    total_differences_save["Model Total"].round(1)
)

total_differences_save["Market Total"] = (
    total_differences_save["Market Total"].round(1)
)

total_differences_save["Total Diff"] = (
    total_differences_save["Total Diff"].round(1)
)

# Save outputs
projection_board_save.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{WEEK_NUMBER:02d}_projection_board.csv",
    index=False
)

ats_differences_save.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{WEEK_NUMBER:02d}_biggest_ats_differences.csv",
    index=False
)

total_differences_save.to_csv(
    WEEK_OUTPUT_DIR
    / f"week_{WEEK_NUMBER:02d}_biggest_total_differences.csv",
    index=False
)

print("Saved:")
print(
    WEEK_OUTPUT_DIR
    / f"week_{WEEK_NUMBER:02d}_projection_board.csv"
)
print(
    WEEK_OUTPUT_DIR
    / f"week_{WEEK_NUMBER:02d}_biggest_ats_differences.csv"
)
print(
    WEEK_OUTPUT_DIR
    / f"week_{WEEK_NUMBER:02d}_biggest_total_differences.csv"
)

Saved:
..\..\weekly_projections\outputs\week_01\week_01_projection_board.csv
..\..\weekly_projections\outputs\week_01\week_01_biggest_ats_differences.csv
..\..\weekly_projections\outputs\week_01\week_01_biggest_total_differences.csv
